In [1]:
import numpy as np
from pytenet.global_krylov_method import  get_W, get_S
from pytenet.global_krylov_method import generate_krylov_space, generate_reduced_H_non_ortho,generate_krylov_space_test_poly_with_constant
import numpy as np
import copy
import h5py
from pytenet.mpo import MPO, add_mpo
from pytenet.mps import MPS, create_random_mps,create_random_mps_4

import numpy.random as random
np.set_printoptions(precision=4,suppress=True)
import matplotlib.pyplot as plt
import pickle
from pytenet.hamiltonian_thc import eval_func
import pytenet as ptn
from pytenet.operation import outer_product,vdot,apply_operator, add_mps_and_compress,add_mps, apply_operator_and_compress,apply_operator, operator_average,operator_inner_product
from scipy.sparse.linalg import eigsh
from pytenet.Lanczos import dmrg_singlesite,dmrg_twosite,solve_ritz_vecs,generate_re_ortho_space, generate_restart_with_ritz,generate_Krylov_space_TRL ,generate_band_krylov,generate_single_state_bose,generate_single_state_spin,generate_single_state_spin1
import copy
import scipy
import pytenet.mps
from matplotlib.ticker import FuncFormatter

In [2]:
#load integrals
with h5py.File("data_water/eri_water.hdf5", "r") as f:
#with h5py.File("/work_fast/ge49cag/pytenet_yu/water/eri_water.hdf5", "r") as f:
    eri = f["eri"][()]
    hkin = f["hkin"][()]
    hnuc = f["hnuc"][()]

#print(np.linalg.norm(eri))
#print(eri.shape)

no = eri.shape[0]
MV = eri.reshape(no*no,no*no)

u = np.load("data_water/x.npy")
#u = np.load("/work_fast/ge49cag/pytenet_yu/water/x.npy")
X_mo = u.transpose(1,0)
g_thc, Z_mo = eval_func(u,eri,hkin,hnuc,)
h1 = hnuc+hkin
nmo = X_mo.shape[1]
L = X_mo.shape[1]
g_thc = g_thc.reshape(nmo, nmo, nmo, nmo)
r_thc = X_mo.shape[0]
g_phy =  eri.transpose(0, 2, 1, 3)
#mpo_ref = ptn.hamiltonian.molecular_hamiltonian_mpo(h1_spin, g_spin_phy)
mpo_ref = ptn.hamiltonian.spin_molecular_hamiltonian_mpo(h1, g_phy)
matrix = mpo_ref.as_matrix(sparse_format=True)
realeig, real_vec = eigsh(matrix, k=20, which='SA', return_eigenvectors=True)
print(realeig)

7
(7, 28)
(28, 28)
rl errV: 2.5804847549658373e-12
abs errV: 1.874062745450254e-11
errt: 2.6607105502789027e-13
errh: 2.300673316482739e-13
errht: 2.100739977085022e-13
[-84.9226 -84.5989 -84.5989 -84.4682 -84.5278 -84.5099 -84.5099 -84.3867
 -84.5278 -84.5278 -84.4249 -84.4249 -84.4238 -84.4238 -84.3285 -84.4249
 -84.4238 -84.3481 -84.3481 -84.3481]


In [3]:
e_ground = -84.92262983120877
e_1st_ex = -84.52780562388604
e_2nd_ex = -84.46820215626565
e_3rd_ex = -84.42486181064207
e_4th_ex = -84.42379006099287
realeig =[ e_ground, e_1st_ex, e_2nd_ex, e_3rd_ex, e_4th_ex]
realeig

[-84.92262983120877,
 -84.52780562388604,
 -84.46820215626565,
 -84.42486181064207,
 -84.42379006099287]

In [ ]:
opt_gs_err = []
opt_1st_err = []
opt_2nd_err = []
opt_3rd_err = []
opt_4th_err = []
#bond_dim_list = [1,6,10,20,30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,186]
bond_dim_list = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33]
for i in bond_dim_list:
    gs_vec = copy.deepcopy(real_vec.T[0])
    ex1st_vec = copy.deepcopy(real_vec.T[6])
    ex2nd_vec = copy.deepcopy(real_vec.T[4])
    ex3rd_vec = copy.deepcopy(real_vec.T[11])
    ex4th_vec = copy.deepcopy(real_vec.T[7])
    gs_mps = ptn.MPS.from_vector(4,7,gs_vec)
    ex1st_mps = ptn.MPS.from_vector(4,7,ex1st_vec)
    ex2nd_mps = ptn.MPS.from_vector(4,7,ex2nd_vec)
    ex3rd_mps = ptn.MPS.from_vector(4,7,ex3rd_vec)
    ex4th_mps = ptn.MPS.from_vector(4,7,ex4th_vec)
    gs_mps.orthonormalize("right")
    ex1st_mps.orthonormalize("right")
    ex2nd_mps.orthonormalize("right")
    ex3rd_mps.orthonormalize("right")
    ex4th_mps.orthonormalize("right")
    gs_mps.compress_direct_svd_left_max_bond(tol=0,max_bond=i)
    gs_mps.orthonormalize("right")
    gs_mps.orthonormalize("left")
    ex1st_mps.compress_direct_svd_left_max_bond(tol=0,max_bond=i)
    ex1st_mps.orthonormalize("right")
    ex1st_mps.orthonormalize("left")
    ex2nd_mps.compress_direct_svd_left_max_bond(tol=0,max_bond=i)
    ex2nd_mps.orthonormalize("right")
    ex2nd_mps.orthonormalize("left")
    ex3rd_mps.compress_direct_svd_left_max_bond(tol=0,max_bond=i)
    ex3rd_mps.orthonormalize("right")
    ex3rd_mps.orthonormalize("left")
    ex4th_mps.compress_direct_svd_left_max_bond(tol=0,max_bond=i)
    ex4th_mps.orthonormalize("right")
    ex4th_mps.orthonormalize("left")

    opt_gs_err.append(np.linalg.norm(operator_average(gs_mps, mpo_ref) - realeig[0])    )
    opt_1st_err.append(np.linalg.norm(operator_average(ex1st_mps, mpo_ref) - realeig[6])  )
    opt_2nd_err.append(np.linalg.norm(operator_average(ex2nd_mps, mpo_ref) - realeig[4])  )
    opt_3rd_err.append(np.linalg.norm(operator_average(ex3rd_mps, mpo_ref) - realeig[11])  )
    opt_4th_err.append(np.linalg.norm(operator_average(ex4th_mps, mpo_ref) - realeig[7])   )

###(1.8053718520150142e-09+0j) for 30

In [ ]:
plt.figure(figsize=(8, 6),dpi = 100,constrained_layout=True)
plt.plot(bond_dim_list,opt_gs_err)
plt.plot(bond_dim_list,opt_1st_err)
plt.plot(bond_dim_list,opt_2nd_err)
plt.plot(bond_dim_list,opt_3rd_err)
plt.plot(bond_dim_list,opt_4th_err)
plt.legend(["GS","1st","2nd","3rd","4th"],fontsize=16)
plt.xlabel("Bond Dimension", fontsize=16)
plt.ylabel("Energy error", fontsize=16)
ax = plt.gca()
ax.tick_params(axis='both', which='major', labelsize=16)
xticks = [i for i in range(0,37,4)]  # 10^0 to 10^-10
plt.xticks(xticks)
plt.yscale("log")

plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\h2o_ed.pdf",bbox_inches='tight',
        pad_inches=0.05)
plt.show()

In [ ]:
### Generate the Heisenberg MPO
#mpo_ref = ptn.heisenberg_xxz_mpo(L = 12, J= -4, D = -4, h= 2)
mpo_ref = ptn.fermi_hubbard_mpo(L = 7, U=8, t=1, mu=1)
#mpo_ref = ptn.bose_hubbard_mpo(d=2,L = 12, U=10, t=1, mu = 1)
print(mpo_ref.bond_dims)
mpo_ref.zero_qnumbers()
matrix = mpo_ref.as_matrix(sparse_format=True)
realeig, real_vec = eigsh(matrix, k=10, which='SA', return_eigenvectors=True)
print(realeig)


In [ ]:
realeig

In [ ]:
opt_err = []
for i in range(5):
    gs_mps = pytenet.MPS.from_vector(d=4, nsites=7, v=real_vec[:,i], tol=0)
    gs_mps.orthonormalize('right')
    gs_mps.compress_direct_svd_left_max_bond(1e-12,20)
    gs_mps.orthonormalize('right')
    opt_err.append(operator_average(gs_mps,mpo_ref)  ) 
opt_err


In [ ]:
### Here we have no truncation, for testing the eigenvalues
N_Krylov = 50
H = mpo_ref
max_bond_Krylov = 5
#max_bond_Krylov = 75
trunc_tol = 0
foldername = f"test"
N_list = []
rng_master = np.random.default_rng(42)
v = create_random_mps([1] * 11, 'right', rng=rng_master)
generate_krylov_space(N_Krylov, H, v, max_bond_Krylov, trunc_tol, foldername) 
N_use = 50
# C = coeff_canonical_orthogonalization(N_use, foldername)
# vector_list = generate_re_ortho_space_with_coeff(N_use, C, foldername)
#vector_list = generate_re_ortho_space(N_use, foldername)
H_reduced_non_rotho = generate_reduced_H_non_ortho(N_use, foldername, mpo_ref)
coeff_W = get_W(N_use, foldername)
coeff = get_S(coeff_W)
coeff = np.array(coeff)
H_reduced = np.einsum('ik, kl, jl -> ij', coeff.conj(), H_reduced_non_rotho, coeff)
e = np.linalg.eigvalsh(H_reduced)
e

In [ ]:
def lanczos_lr_py(H, v0_mps, maxiter, tol, rmax):
    """
    Python implementation of the rank-truncated Lanczos algorithm, mirroring the
    logic of the Dektor et al. MATLAB script.

    This function generates a non-orthogonal basis and solves the generalized
    eigenvalue problem <v|H|v> = lambda * <v|v> at each step.

    Args:
        H (MPO): The Hamiltonian operator.
        v0_mps (MPS): The initial, normalized MPS vector.
        maxiter (int): The maximum number of Lanczos iterations.
        tol (float): The truncation tolerance for SVD.
        rmax (int): The maximum allowed bond dimension.

    Returns:
        V (list): A list of the generated non-orthogonal MPS Krylov vectors.
        T (np.ndarray): The final tridiagonal matrix (for comparison).
        eigs (list): A list where each element contains the eigenvalues at that iteration.
    """
    # --- Initialization ---
    alpha = []
    # In MATLAB, beta starts with 1 element, then grows.
    # We will append the norm before normalization.
    beta = [0.0] 
    
    eigs = []
    V = [copy.deepcopy(v0_mps)]

    # --- First Step (j=1) ---
    # Corresponds to MATLAB lines 14-22
    
    # 1. Apply operator and truncate
    # v_next corresponds to V{2} in MATLAB before orthogonalization
    v_next = apply_operator_and_compress(H, V[0], tol, rmax)
    
    # 2. Calculate alpha_0
    alpha.append(vdot(v_next, V[0]))

    # 3. Orthogonalize against v_0 and truncate
    temp = copy.deepcopy(V[0])
    # Scale MPS by multiplying the first tensor
    temp.A[0] = -alpha[0] * temp.A[0]
    v_next = add_mps_and_compress(v_next, temp, tol, rmax)
    
    # V[1] is now the un-normalized result of the first recurrence
    V.append(v_next)
    
    # 4. Initialize the W and P matrices
    W = np.array([[vdot(V[0], V[0])]], dtype=np.complex128)
    P = np.array([[operator_inner_product(V[0], H, V[0])]], dtype=np.complex128)
    
    print(f"Lanczos Iteration 1: Eigenvalues = {np.linalg.eigvalsh(P)}")
    eigs.append(np.sort(np.real(np.linalg.eigvalsh(P))))
    
    # --- Main Loop ---
    # Corresponds to MATLAB lines 23-40
    for j in range(1, maxiter):
        
        # 5. Calculate beta_j and normalize V[j]
        # V[j] is the un-normalized vector from the previous step
        current_beta = V[j].orthonormalize('right')
        if current_beta < 1e-14:
            print("Encountered zero-norm vector. Stopping.")
            break
        beta.append(current_beta)
        
        # Normalize V[j] in-place
        #V[j].A[0] /= current_beta

        # 6. Apply operator to the new normalized vector
        v_next = apply_operator_and_compress(H, V[j], tol, rmax)
        
        # 7. Calculate alpha_j
        alpha.append(vdot(v_next, V[j]))

        # 8. Three-term recurrence with sequential truncation
        # Subtract alpha_j * v_j term
        temp1 = copy.deepcopy(V[j])
        temp1.A[0] = -alpha[j] * temp1.A[0]
        v_next = add_mps_and_compress(v_next, temp1, tol, rmax)
        
        # Subtract beta_j * v_{j-1} term
        temp2 = copy.deepcopy(V[j-1])
        temp2.A[0] = -beta[j] * temp2.A[0]
        v_next = add_mps_and_compress(v_next, temp2, tol, rmax)
        
        # Store the new un-normalized vector
        V.append(v_next)

        # 9. Update W and P matrices by adding a new row/column
        # This is more efficient than rebuilding the matrices every time
        new_dim = j + 1
        W_new = np.zeros((new_dim, new_dim), dtype=np.complex128)
        P_new = np.zeros((new_dim, new_dim), dtype=np.complex128)
        W_new[:j, :j] = W
        P_new[:j, :j] = P
        
        for k in range(new_dim):
            W_new[j, k] = vdot(V[j], V[k])
            W_new[k, j] = np.conj(W_new[j, k])
            
            P_new[j, k] = operator_inner_product(V[j], H, V[k])
            P_new[k, j] = np.conj(P_new[j, k])
        
        W, P = W_new, P_new

        # 10. Solve the Generalized Eigenvalue Problem
        try:
            current_eigs, _ = scipy.linalg.eigh(P, W)
            eigs.append(np.sort(np.real(current_eigs)))
        except np.linalg.LinAlgError:
            print(f"Warning: Generalized eigenvalue solve failed at iteration {j+1}. W may be singular.")
            eigs.append(eigs[-1]) # Reuse previous eigenvalues

        # 11. Diagnostic print
        if (j + 1) % 10 == 0:
            print(f"Lanczos Iteration {j+1}: \t Condition # of basis: {np.linalg.cond(W):.3e}")
            print(f"Lowest eigenvalue: {eigs[-1][0]}")

    
    return V,P,W,eigs

In [ ]:
from pytenet.Lanczos import ortho_to_list
def band_lanczos(H, initial_block, num_new_vecs, tol, rmax, lock_idx=0):
    """
    Python implementation of a Band Lanczos / MRL algorithm with
    SELECTIVE RE-ORTHOGONALIZATION.

    Each new vector is only orthogonalized against the 'l' most recent vectors
    in the basis, where 'l' is the size of the initial block.

    Args:
        H (MPO): The Hamiltonian operator.
        initial_block (list): A list of starting, normalized MPS vectors.
        num_new_vecs (int): The number of new Krylov vectors to generate.
        tol (float): The truncation tolerance for SVD.
        rmax (int): The maximum allowed bond dimension.
        lock_idx (int): The number of initial vectors to "lock" (not use for generation).

    Returns:
        V (list): A list of the generated non-orthogonal MPS Krylov vectors.
        P (np.ndarray): The final projected Hamiltonian matrix in the non-orthogonal basis.
        W (np.ndarray): The final Gram (overlap) matrix.
        eigs (list): A list where each element contains the eigenvalues at that iteration.
    """
    if not initial_block:
        raise ValueError("Initial block cannot be empty.")
    
    # 'l' is the width of the band (the size of the initial block)
    l = len(initial_block)
    V = copy.deepcopy(initial_block)
    eigs = []
    
    # Initialize the W and P matrices from the starting block
    W = np.zeros((l, l))
    P = np.zeros((l, l))
    for i in range(l):
        for k in range(i, l):
            W[i, k] = vdot(V[i], V[k])
            W[k, i] = np.conj(W[i, k])
            P[i, k] = operator_inner_product(V[i], H, V[k])
            P[k, i] = np.conj(P[i, k])
    
    try:
        current_eigs, _ = scipy.linalg.eigh(P, W)
        eigs.append(np.sort(np.real(current_eigs)))
        print(f"Initial block (size {l}): Lowest eigenvalue = {eigs[-1][0]}")
    except np.linalg.LinAlgError:
        print("Error: Initial block is linearly dependent.")
        return V, None, None, []

    # --- Main Loop to generate new vectors ---
    for i in range(num_new_vecs):
        # 1. Select a vector to expand from
        seed_idx = lock_idx + (i % (l - lock_idx))
        seed_vector = V[seed_idx]
        print(f"\n--- Iteration {i+1}/{num_new_vecs} (expanding from vector {seed_idx}) ---")

        # 2. Generate new candidate vector
        v_next = apply_operator_and_compress(H, seed_vector, tol, rmax)

        # 3. --- MODIFICATION HERE: SELECTIVE ORTHOGONALIZATION ---
        # Orthogonalize the new vector against only the last '2l' vectors in the basis.
        # This is a fixed-window Gram-Schmidt procedure.
        
        # Determine the slice of the basis to orthogonalize against
        ortho_basis = V[-2*l+1:]
        
        v_next = ortho_to_list(ortho_basis, v_next, rmax, tol) 

        
        # 4. Normalize the new vector
        current_norm = v_next.orthonormalize('right')
        if current_norm < 1e-14:
            print("Warning: New vector has nearly zero norm. Subspace may have converged.")
            continue
    
        V.append(v_next)
        
        # 5. Update W and P matrices (still done on the FULL basis for accuracy)
        current_basis_size = len(V)
        new_dim = current_basis_size
        W_new = np.zeros((new_dim, new_dim), dtype=np.complex128)
        P_new = np.zeros((new_dim, new_dim), dtype=np.complex128)
        W_new[:new_dim-1, :new_dim-1] = W
        P_new[:new_dim-1, :new_dim-1] = P

        for k in range(new_dim):
            W_new[new_dim-1, k] = vdot(V[-1], V[k])
            W_new[k, new_dim-1] = np.conj(W_new[new_dim-1, k])
            P_new[new_dim-1, k] = operator_inner_product(V[-1], H, V[k])
            P_new[k, new_dim-1] = np.conj(P_new[new_dim-1, k])
        
        W, P = W_new, P_new

        # 6. Solve the Generalized Eigenvalue Problem
        try:
            current_eigs, _ = scipy.linalg.eigh(P, W)
            eigs.append(np.sort(np.real(current_eigs)))
        except np.linalg.LinAlgError:
            print(f"Warning: Generalized eigenvalue solve failed at iteration {i+1}. W may be singular.")
            eigs.append(eigs[-1])

        # 7. Diagnostic print
        print(f"Basis size {new_dim}: \t Condition # of W: {np.linalg.cond(W):.3e}")
        print(f"Lowest eigenvalue: {eigs[-1][0]}")

    return V, P, W, eigs

In [ ]:
def solve_ritz_vecs_generalized(folder_containing_Krylov, P, W, n, max_bond, num_ritz, mpo_ref):
    """
    Constructs the Ritz vector MPSs using the stable generalized eigenvalue method.

    Args:
        folder_containing_Krylov (str): Foldername which contains Krylov vectors.
        P (np.ndarray): The projected Hamiltonian in the non-orthogonal basis (<v_i|H|v_j>).
        W (np.ndarray): The Gram matrix of the non-orthogonal basis (<v_i|v_j>).
        n (int): The current size of the Krylov subspace to use.
        max_bond (int): The max_bond used to represent the Ritz vector.
        num_ritz (int): The number of Ritz vectors to calculate.
        mpo_ref (MPO): MPO used to calculate the final energy expectation value.

    Returns:
        test_energy (list): The list of calculated Ritz values (energies).
        krylov_vec (list): A list of the explicitly constructed Ritz vector MPS objects.
    """
    # Step 1: Solve the generalized eigenvalue problem. This is the core change.
    # We solve P*v = e*W*v. The columns of 'v' are the coefficients we need.
    try:
        # We only need the first 'n' vectors from the basis
        e, v = scipy.linalg.eigh(P[:n, :n], W[:n, :n])
    except np.linalg.LinAlgError:
        print("Error: Generalized eigenvalue problem did not converge. W may be singular.")
        return [], []

    ritz_vecs = []
    ritz_vals = []

    print(f"\n--- Constructing {num_ritz} Ritz Vectors from a basis of size {n} ---")

    # Step 2: Loop to construct each of the desired Ritz vectors
    for count in range(num_ritz):
        
        # The k-th column of v gives the coefficients for the k-th Ritz vector.
        # There is no need to multiply by a 'coeff' matrix.
        C_ritz = v[:, count]
        
        # Step 3: Explicitly construct the Ritz vector MPS by summing the basis vectors.
        # NOTE: This part is still computationally expensive and introduces truncation error,
        # but it is necessary if you need the explicit MPS object.
        
        # Load the first basis vector to start the sum
        filename = folder_containing_Krylov + f"/Krylov_vec{0}.pkl"
        with open(filename, 'rb') as file:
            # Use deepcopy to avoid modifying the stored vectors
            ritz_vec = copy.deepcopy(pickle.load(file))
        ritz_vec.A[0] = C_ritz[0] * ritz_vec.A[0]
        
        # Add the remaining weighted basis vectors
        for i in range(1, n):
            filename = folder_containing_Krylov + f"/Krylov_vec{i}.pkl"
            with open(filename, 'rb') as file:
                temp = copy.deepcopy(pickle.load(file))
            
            temp.A[0] = C_ritz[i] * temp.A[0]
            
            # This is the expensive summation and compression step
            ritz_vec = add_mps_and_compress(ritz_vec, temp, 0, 10 * max_bond)

        # Final compression to the target bond dimension
        ritz_vec.compress_direct_svd_right_max_bond(0, max_bond)
        ritz_vec.orthonormalize('right')
        
        # Calculate the expectation value to verify the energy
        e_ritz = operator_average(ritz_vec, mpo_ref)
        
        ritz_vecs.append(ritz_vec)
        # The eigenvalue from the small matrix problem is the more accurate one
        ritz_vals.append(e[count])
        
        print(f"Constructed Ritz vector {count}. Ritz value: {e[count]:.12f}, <H>: {e_ritz:.12f}")

    return ritz_vals, ritz_vecs

In [ ]:
def solve_ritz_vecs_from_list(V_basis, P, W, n, max_bond, num_ritz, mpo_ref):
    """
    Constructs the Ritz vector MPSs from a list of basis vectors using the
    stable generalized eigenvalue method.

    Args:
        V_basis (list): A list containing the non-orthogonal MPS basis vectors.
        P (np.ndarray): The projected Hamiltonian in the non-orthogonal basis (<v_i|H|v_j>).
        W (np.ndarray): The Gram matrix of the non-orthogonal basis (<v_i|v_j>).
        n (int): The current size of the Krylov subspace to use.
        max_bond (int): The max_bond used to represent the Ritz vector.
        num_ritz (int): The number of Ritz vectors to calculate.
        mpo_ref (MPO): MPO used to calculate the final energy expectation value.

    Returns:
        ritz_vals (list): The list of calculated Ritz values (energies).
        ritz_vecs (list): A list of the explicitly constructed Ritz vector MPS objects.
    """
    # Step 1: Solve the generalized eigenvalue problem.
    # The columns of 'v' are the coefficients for the non-orthogonal V_basis.
    try:
        e, v = scipy.linalg.eigh(P[:n, :n], W[:n, :n])
    except np.linalg.LinAlgError:
        print("Error: Generalized eigenvalue problem did not converge. W may be singular.")
        return [], []

    ritz_vecs = []
    ritz_vals = []

    print(f"\n--- Constructing {num_ritz} Ritz Vectors from a basis of size {n} ---")

    # Step 2: Loop to construct each of the desired Ritz vectors
    for count in range(num_ritz):
        
        # The k-th column of v gives the coefficients for the k-th Ritz vector.
        C_ritz = v[:, count]
        
        # Step 3: Explicitly construct the Ritz vector MPS by summing the basis vectors.
        
        # --- MODIFICATION START ---
        # Load the first basis vector from the list instead of a file
        # Use deepcopy to avoid modifying the original basis vectors in the list
        ritz_vec = copy.deepcopy(V_basis[0])
        # --- MODIFICATION END ---
        
        ritz_vec.A[0] = C_ritz[0] * ritz_vec.A[0]
        
        # Add the remaining weighted basis vectors
        for i in range(1, n):
            # --- MODIFICATION START ---
            # Load the i-th basis vector from the list
            temp = copy.deepcopy(V_basis[i])
            # --- MODIFICATION END ---
            
            temp.A[0] = C_ritz[i] * temp.A[0]
            
            # This is the expensive summation and compression step
            ritz_vec = add_mps_and_compress(ritz_vec, temp, 0,  10 * max_bond)


        ritz_vec.orthonormalize('right')
        
        # Calculate the expectation value to verify the energy
        e_ritz = operator_average(ritz_vec, mpo_ref)
        
        ritz_vecs.append(ritz_vec)
        # The eigenvalue from the small matrix problem is the more accurate one
        ritz_vals.append(e[count])
        
        print(f"Constructed Ritz vector {count}. Ritz value: {e[count]:.12f}, <H>: {e_ritz:.12f}")

    return ritz_vals, ritz_vecs

In [ ]:
r0 = copy.deepcopy(v)
v0,P0,W0,eigs0 = lanczos_lr_py(H,r0, 50, trunc_tol, max_bond_Krylov)
#e_ritz, ritz_vecs = solve_ritz_vecs_from_list(v0, P0, W0, 50, max_bond_Krylov,10,H)

In [ ]:
eigs0[-1]

In [ ]:
import numpy as np

def get_S0(W):
    """
    Correctly implements the recursive reorthogonalization from the appendix of
    Phys. Rev. B 85, 205119, adapted for complex arithmetic.

    Args:
        W (np.ndarray): The complex, Hermitian overlap matrix W_ij = <f_i|f_j>.

    Returns:
        np.ndarray: The final transformation matrix S.
    """
    W = np.asarray(W, dtype=np.complex128)
    n_krylov = W.shape[0]

    # Initialize S and S_tilde as empty matrices. We will build them column by column.
    S = np.zeros_like(W, dtype=np.complex128)
    S_tilde = np.zeros_like(W, dtype=np.complex128)

    # Base case for n=0
    S_tilde[0, 0] = 1.0
    # The norm N_0 is sqrt(S_tilde_0 * W * S_tilde_0) = sqrt(1*W_00*1).
    # Assuming the first vector is normalized, W_00 = 1.
    N0 = np.sqrt(np.real(W[0, 0]))
    S[:, 0] = S_tilde[:, 0] / N0

    # Main recursion loop to build columns n=1, 2, ...
    for n in range(1, n_krylov):
        # This loop implements Eq. (A2) from the paper's appendix.

        # 1. Calculate the intermediate overlap terms k_in = <ψ_i|f_n>
        # k_in = Σ_k S_ki* W_kn. This uses the final, normalized S columns.
        k = np.zeros(n, dtype=np.complex128)
        for i in range(n):
            # Correct complex inner product: S_i^dagger @ W_n
            k[i] = np.dot(np.conj(S[:, i]), W[:, n])

        # 2. Build the un-normalized column S̃_n
        S_tilde[n, n] = 1.0  # For p = n
        for p in range(n):   # For p < n
            # Recursion: S̃_pn = -Σ_{q=p}^{n-1} k_qn * S_pq
            # S_pq is the p-th component of the q-th S vector
            term = 0.0
            for q in range(p, n):
                term += k[q] * S[p, q]
            S_tilde[p, n] = -term

        # 3. Calculate the norm N_n of the new un-normalized vector S̃_n
        # N_n^2 = S̃_n^dagger * W * S̃_n. This is the corrected norm formula.
        norm_sq = np.dot(np.conj(S_tilde[:, n]), W @ S_tilde[:, n])
        Nn = np.sqrt(np.real(norm_sq))

        # 4. Normalize the column to get the final S_n
        if Nn > 1e-14:
            S[:, n] = S_tilde[:, n] / Nn
        # else: S[:, n] remains zeros, indicating linear dependence.

    return S





def get_W1(N_use, V0):
    # Initialize the matrix with the complex data type
    W = np.zeros([N_use, N_use], dtype=np.complex128)

    for i in range(N_use):
        for j in range(i, N_use):  # Loop only over the upper triangle (j >= i)
            # Calculate the upper-triangle element
            overlap = vdot(V0[i], V0[j])
            W[i, j] = overlap
            
            # Enforce the Hermitian property for the lower-triangle element
            if i != j:
                W[j, i] = np.conj(overlap)
    
    return W


def generate_reduced_H(N, V0, H_mpo):
    # Initialize the matrix with the complex data type
    H_reduced = np.zeros([N, N], dtype=np.complex128)

    for i in range(N):
        for j in range(i, N): # Loop only over the upper triangle (j >= i)
            # Calculate the upper-triangle element
            element = operator_inner_product(V0[i], H_mpo, V0[j])
            H_reduced[i, j] = element
            
            # Enforce the Hermitian property for the lower-triangle element
            if i != j:
                H_reduced[j, i] = np.conj(element)
                
    return H_reduced




In [ ]:
test_N_Krylov = 50
trunc_tol = 1e-13
max_bond_Krylov = 300
rng_master = np.random.default_rng(42)
#test_H = ptn.heisenberg_xxz_mpo(16,1,1,0)
#test_H = ptn.ising_mpo(16, 1, 1, 1)
test_H = ptn.fermi_hubbard_mpo(L=8, U=8, t=1, mu=1)
test_HFS = create_random_mps_4([1] * 9, 'right', rng=rng_master)
#test_HFS =  ptn.generate_single_state(8, [3, 3, 3, 3, 1, 2, 0,3])
test_HFS.zero_qnumbers()
test_H.zero_qnumbers()

v0,P0,W0,eigs0 = lanczos_lr_py(test_H,test_HFS, 50, trunc_tol, max_bond_Krylov)
krylov1 = copy.deepcopy(v0[-1])
krylov1.orthonormalize('right')
bond = [20,30,40,50,60,70,80,90,100]
bond_test = []
for b in bond:
    v = copy.deepcopy(krylov1)
    v.orthonormalize('right')
    v.compress_direct_svd_left_max_bond(tol = 1e-14, max_bond=b)
    v.orthonormalize('right')
    bond_test.append(np.abs(vdot(v,krylov1))**2)


In [ ]:
bond_test1 = []
for b in bond:
    v0,P0,W0,eigs0 = lanczos_lr_py(test_H,test_HFS, 50, trunc_tol, b)
    krylov2 = v0[-1]
    krylov2.orthonormalize('right')
    bond_test1.append(np.abs(vdot(krylov2,krylov1))**2)

In [ ]:
plt.figure(figsize=(8, 6),dpi=100)

plt.plot(bond, bond_test, 'o-', label="Direct Truncation")
plt.plot(bond, bond_test1, '^-', label="Successive Truncation")


plt.xlabel("Bond Dimension", fontsize=16)
plt.ylabel("Fidelity", fontsize=16)


ax = plt.gca()
ax.tick_params(axis='both', which='major', labelsize=16)

# Add horizontal grid line at 1e-7

# Set log scale
ax.set_yscale('log')


plt.legend(fontsize = 12)
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\bond_fermi.pdf",bbox_inches='tight',
        pad_inches=0.05)
plt.show()


In [ ]:
#energy_list = [eigs0[-1][:5]]  # Store the first Ritz value as the initial energy
rng_master = np.random.default_rng(42)


block = []
for i in range(5):
    print(f"Generating MPS #{i+1}...")
    #new_mps = create_random_mps([1,2,2,2,2,2,2,2,2,2,2,2,1], 'right', rng=rng_master)
    new_mps = create_random_mps_4([1, 4, 16, 20, 20, 16, 4, 1], 'right', rng=rng_master)
    block.append(new_mps)
energy_list = []

for restart in range(350):
    print(f"\n--- Restart {restart + 1} ---")
    v0,P0,W0,eig0 = band_lanczos(mpo_ref, block[:5], 20, 1e-12, 20)
    e_ritz, block = solve_ritz_vecs_from_list(v0, P0, W0,25, 20, 5, mpo_ref)
    energy_list.append(eig0[-1][:5])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8), dpi=100)
k = 5
realeig = np.sort(realeig)  

N_list = np.linspace(4, 4+349*4, 350)
# Define a color cycle for the plots
colors = plt.cm.viridis(np.linspace(0, 0.9, k))

# --- Calculate Errors and Plot ---
state_labels = ["Ground State 1", "Ground State 2", "First Excited State 1", "First Excited State 2", "Third Excited State"]

for i in range(k):
    # Calculate the absolute error for the i-th state at each step
    error_values = [np.abs(step_energies[i] - realeig[i]) for step_energies in energy_list]
    
    # Plot the error convergence line
    ax.plot(
        N_list, 
        error_values, 
        'o-', 
        label=state_labels[i],
        color=colors[i]
    )
    
    # Plot the optimized error horizontal line with the same color
    ax.axhline(
            y=np.abs(opt_err[i]),
            color=colors[i],
            linestyle='--',
            linewidth=2,
            # The label for hlines can make the legend crowded, so it's often omitted
            # or handled differently if needed.
            # label=f'Optimized Error: {state_labels[i]}' 
        )

# --- Formatting and Labels ---
ax.set_yscale('log')
ax.set_xlabel("Number of Krylov Vectors for Each Ritz Vector", fontsize=16)
ax.set_ylabel(" Energy Error", fontsize=16)
ax.set_ylim(1e-6, 1e2)

ax.tick_params(axis='both', which='major', labelsize=16)
#ax.grid(True, which='both', linestyle='--', linewidth=0.5)

# Use the y-tick formatter from your original script
yticks = [10**(-i) for i in range(0, 7, 2)] # up to 10^-6
ax.set_yticks(yticks)
formatter = FuncFormatter(lambda y, _: f"$10^{{{int(np.log10(y))}}}$")
ax.yaxis.set_major_formatter(formatter)

ax.legend(fontsize=14)

fig.tight_layout()

# --- Save and Show the Plot ---
# To save the figure, provide a valid path
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\hei_mrl_600.pdf", bbox_inches='tight', pad_inches=0.05)

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8), dpi=100)
k = 5


N_list = np.linspace(4, 4+349*4, 350)
# Define a color cycle for the plots
colors = plt.cm.viridis(np.linspace(0, 0.9, k))


error_values = [np.abs(step_energies[0] - realeig[0]) for step_energies in energy_list]
    
    # Plot the error convergence line
ax.plot(
        N_list, 
        error_values, 
        'o-', 
        label='Ground State 1',
        color=colors[0]
    )
    
    # Plot the optimized error horizontal line with the same color
ax.axhline(
            y=np.abs(opt_err[0]),
            color=colors[0],
            linestyle='--',
            linewidth=2,
            # The label for hlines can make the legend crowded, so it's often omitted
            # or handled differently if needed.
            # label=f'Optimized Error: {state_labels[i]}' 
        )

error_values = [np.abs(step_energies[1] - realeig[1]) for step_energies in energy_list]
    
    # Plot the error convergence line
ax.plot(
        N_list, 
        error_values, 
        'o-', 
        label='Ground State 2',
        color=colors[1]
    )
    
    # Plot the optimized error horizontal line with the same color
ax.axhline(
            y=np.abs(opt_err[1]),
            color=colors[1],
            linestyle='--',
            linewidth=2,
            # The label for hlines can make the legend crowded, so it's often omitted
            # or handled differently if needed.
            # label=f'Optimized Error: {state_labels[i]}' 
        )

error_values = [np.abs(step_energies[2] - realeig[2]) for step_energies in energy_list]
    
    # Plot the error convergence line
ax.plot(
        N_list, 
        error_values, 
        'o-', 
        label='First Excited State 1',
        color=colors[2]
    )
    
    # Plot the optimized error horizontal line with the same color
ax.axhline(
            y=np.abs(opt_err[2]),
            color=colors[2],
            linestyle='--',
            linewidth=2,
            # The label for hlines can make the legend crowded, so it's often omitted
            # or handled differently if needed.
            # label=f'Optimized Error: {state_labels[i]}' 
        )


error_values = [np.abs(step_energies[3] - realeig[3]) for step_energies in energy_list]
    
    # Plot the error convergence line
ax.plot(
        N_list, 
        error_values, 
        'o-', 
        label='First Excited State 2',
        color=colors[3]
    )
    
    # Plot the optimized error horizontal line with the same color
ax.axhline(
            y=np.abs(opt_err[3]),
            color=colors[3],
            linestyle='--',
            linewidth=2,
            # The label for hlines can make the legend crowded, so it's often omitted
            # or handled differently if needed.
            # label=f'Optimized Error: {state_labels[i]}' 
        )

error_values = [np.abs(step_energies[4] - realeig[5]) for step_energies in energy_list]
    
    # Plot the error convergence line
ax.plot(
        N_list, 
        error_values, 
        'o-', 
        label = 'Third Excited State',
        color=colors[i]
    )
    
    # Plot the optimized error horizontal line with the same color
ax.axhline(
            y=np.abs(opt_err[5]),
            color=colors[4],
            linestyle='--',
            linewidth=2,
            # The label for hlines can make the legend crowded, so it's often omitted
            # or handled differently if needed.
            # label=f'Optimized Error: {state_labels[i]}' 
        )

# --- Formatting and Labels ---
ax.set_yscale('log')
ax.set_xlabel("Number of Krylov Vectors for Each Eigenvector", fontsize=16)
ax.set_ylabel("Absolute energy Error", fontsize=16)
ax.set_ylim(1e-6, 1e2)

ax.tick_params(axis='both', which='major', labelsize=16)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)

# Use the y-tick formatter from your original script
yticks = [10**(-i) for i in range(0, 7, 2)] # up to 10^-6
ax.set_yticks(yticks)
formatter = FuncFormatter(lambda y, _: f"$10^{{{int(np.log10(y))}}}$")
ax.yaxis.set_major_formatter(formatter)

ax.legend(fontsize=14)

fig.tight_layout()

# --- Save and Show the Plot ---
# To save the figure, provide a valid path
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\fermi_mrl_600.pdf", bbox_inches='tight', pad_inches=0.05)

plt.show()


In [ ]:
def trl_plus_r_solver(
    H_mpo,
    k,
    max_bond,
    initial_guess_block,
    realeig,
    num_restarts=20,
    lanczos_steps=10, # Number of new vectors to add per restart
    tol=1e-12,
    trunc_tol=1e-12
):
    """
    Finds the 'k' lowest eigenpairs of a Hamiltonian MPO using a
    Thick Restart Lanczos with Residual Enrichment (TRL+R) algorithm.

    This implementation follows the correct TRL procedure where the basis
    is augmented by a new Lanczos run seeded by the residuals.

    Args:
        H_mpo (MPO): The Hamiltonian operator.
        k (int): The number of lowest eigenvalues to find.
        max_bond (int): The maximum bond dimension for the MPS.
        initial_guess_block (list): A list of 'k' MPS objects to start with.
        realeig (np.ndarray): The true eigenvalues for error tracking.
        num_restarts (int): The number of macro-restart iterations to perform.
        lanczos_steps (int): The number of new vectors to generate in each restart cycle.
        tol (float): Convergence tolerance for the residual norm.
        trunc_tol (float): Truncation tolerance for MPS operations.

    Returns:
        final_eigs (np.ndarray): The k lowest computed eigenvalues.
        final_vecs (list): A list of k MPS objects for the eigenvectors.
        error_history (np.ndarray): A (num_restarts x k) array of absolute errors.
    """
    
    # Start with the initial guess as our first basis
    V_basis = copy.deepcopy(initial_guess_block)
    error_history = []

    # --- Main Restart Loop ---
    for i_restart in range(num_restarts):
        print(f"\n{'='*20} TRL+R Macro-Iteration {i_restart+1}/{num_restarts} {'='*20}")
        print(f"  Current basis size: {len(V_basis)}")

        # --- Phase 1: Rayleigh-Ritz on the Current Basis ---
        print(f"  Phase 1: Solving Rayleigh-Ritz problem...")
        dim_basis = len(V_basis)
        W = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        P = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        for i in range(dim_basis):
            for j in range(i, dim_basis):
                W[i, j] = vdot(V_basis[i], V_basis[j])
                W[j, i] = np.conj(W[i, j])
                P[i, j] = operator_inner_product(V_basis[i], H_mpo, V_basis[j])
                P[j, i] = np.conj(P[i, j])
        
        try:
            ritz_values, ritz_coeffs = scipy.linalg.eigh(P, W)
        except scipy.linalg.LinAlgError:
            print("  FATAL: Eigenvalue solve failed on current basis. Stopping.")
            break
        
        X_block = [form_linear_combination(V_basis, ritz_coeffs[:, j], max_bond, trunc_tol) for j in range(k)]

        # --- Phase 2: Calculate Residuals & Check Convergence ---
        print("  Phase 2: Calculating residuals...")
        R_block = []
        for j_k in range(k):
            ritz_val = ritz_values[j_k]
            ritz_vec = X_block[j_k]
            H_phi = apply_operator_and_compress(H_mpo, ritz_vec, trunc_tol, max_bond)
            lambda_phi = copy.deepcopy(ritz_vec)
            lambda_phi.A[0] *= -ritz_val
            residual_vec = add_mps_and_compress(H_phi, lambda_phi, trunc_tol, max_bond)
            R_block.append(residual_vec)

        residual_norms = [r.orthonormalize('right') for r in R_block]
        print(f"\n  Convergence Check: Residual norms = {np.array(residual_norms)}")
        
        errors = np.abs(np.sort(ritz_values[:k]) - realeig[:k])
        error_history.append(errors)
        print(f"  Current Errors = {errors}")

        if np.all(np.array(residual_norms) < tol):
            print("\nConvergence criteria met. Stopping.")
            break
        
        if i_restart == num_restarts - 1:
            print("\nMax restarts reached.")
            break

        # --- Phase 3: Augment the Basis (Corrected TRL Step) ---
        print("  Phase 3: Augmenting basis with new Lanczos run from residuals...")
        
        # Start the next basis with the k best Ritz vectors. This is the "Thick" part.
        V_next_basis = copy.deepcopy(X_block)
        
        # Orthonormalize the residual block against the new Ritz vectors and itself
        # to create a set of clean, new directions to start the Lanczos run.
        R_ortho_block = []
        for r_vec in R_block:
            # Orthogonalize against all vectors currently in the next basis
            for v_prev in V_next_basis:
                overlap = vdot(r_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                r_vec = add_mps_and_compress(r_vec, term, trunc_tol, max_bond)
            
            norm = r_vec.orthonormalize('right')
            if norm > 1e-12:
                # Add the new, clean search direction to the basis
                V_next_basis.append(r_vec)
        
        # If all residuals collapsed, the subspace is invariant, so we stop.
        if len(V_next_basis) == k:
            print("  Residuals collapsed into the existing Ritz space. Stopping.")
            break

        # Orthogonalization 
            
        #ortho_window_size = 2 * k
        ortho_window_size = k

        num_generated_so_far = len(V_next_basis) - k
        for j_lan in range(num_generated_so_far, lanczos_steps):
            seed_vec = V_next_basis[-1]
            new_vec = apply_operator_and_compress(H_mpo, seed_vec, trunc_tol, max_bond)
            
            # Define the window of recent vectors to orthogonalize against
            start_index = max(0, len(V_next_basis) - ortho_window_size)
            ortho_basis = V_next_basis[start_index:]
            
            # Orthogonalize against the vectors in the window
            for v_prev in ortho_basis:
                overlap = vdot(new_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                new_vec = add_mps_and_compress(new_vec, term, trunc_tol, max_bond)
                
            norm = new_vec.orthonormalize('right')
            if norm > 1e-12:
                V_next_basis.append(new_vec)
            else:
                print(f"  Lanczos vector {j_lan+1} collapsed. Ending augmentation early.")
                break
        
        V_basis = V_next_basis

    return np.sort(ritz_values[:k]), X_block, np.array(error_history)

def form_linear_combination(basis, coeffs_vec, max_bond, trunc_tol):
    active_indices = np.where(np.abs(coeffs_vec) > 1e-14)[0]
    if len(active_indices) == 0:
        new_mps = copy.deepcopy(basis[0])
        new_mps.A[0] *= 0
        return new_mps
    start_idx = active_indices[0]
    new_mps = copy.deepcopy(basis[start_idx])
    new_mps.A[0] *= coeffs_vec[start_idx]
    for i in active_indices[1:]:
        term = copy.deepcopy(basis[i])
        term.A[0] *= coeffs_vec[i]
        new_mps = add_mps_and_compress(new_mps, term, trunc_tol, max_bond)
    new_mps.orthonormalize('right')
    return new_mps

In [ ]:
def trl_plus_r_solver_variance(
    H_mpo,
    k,
    max_bond,
    initial_guess_block,
    num_restarts,
    lanczos_steps, # Number of new vectors to add per restart
    tol,
    trunc_tol
):
    """
    Finds the 'k' lowest eigenpairs of a Hamiltonian MPO using a
    Thick Restart Lanczos with Residual Enrichment (TRL+R) algorithm.

    This version uses the energy variance (sigma^2) as the error metric,
    removing the dependency on known exact eigenvalues.

    Args:
        H_mpo (MPO): The Hamiltonian operator.
        k (int): The number of lowest eigenvalues to find.
        max_bond (int): The maximum bond dimension for the MPS.
        initial_guess_block (list): A list of 'k' MPS objects to start with.
        num_restarts (int): The number of macro-restart iterations to perform.
        lanczos_steps (int): The number of new vectors to generate in each restart cycle.
        tol (float): Convergence tolerance for the residual norm.
        trunc_tol (float): Truncation tolerance for MPS operations.

    Returns:
        final_eigs (np.ndarray): The k lowest computed eigenvalues.
        final_vecs (list): A list of k MPS objects for the eigenvectors.
        variance_history (np.ndarray): A (num_restarts x k) array of energy variances.
    """
    
    # Start with the initial guess as our first basis
    V_basis = copy.deepcopy(initial_guess_block)
    variance_history = []

    # --- Main Restart Loop ---
    for i_restart in range(num_restarts):
        print(f"\n{'='*20} TRL+R Macro-Iteration {i_restart+1}/{num_restarts} {'='*20}")
        print(f"  Current basis size: {len(V_basis)}")

        # --- Phase 1: Rayleigh-Ritz on the Current Basis ---
        print(f"  Phase 1: Solving Rayleigh-Ritz problem...")
        dim_basis = len(V_basis)
        W = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        P = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        for i in range(dim_basis):
            for j in range(i, dim_basis):
                W[i, j] = vdot(V_basis[i], V_basis[j])
                W[j, i] = np.conj(W[i, j])
                P[i, j] = operator_inner_product(V_basis[i], H_mpo, V_basis[j])
                P[j, i] = np.conj(P[i, j])
        
        try:
            ritz_values, ritz_coeffs = scipy.linalg.eigh(P, W)
        except scipy.linalg.LinAlgError:
            print("  FATAL: Eigenvalue solve failed on current basis. Stopping.")
            break
        
        X_block = [form_linear_combination(V_basis, ritz_coeffs[:, j], max_bond, trunc_tol) for j in range(k)]

        # --- Phase 2: Calculate Residuals & Check Convergence ---
        print("  Phase 2: Calculating residuals...")
        R_block = []
        for j_k in range(k):
            ritz_val = ritz_values[j_k]
            ritz_vec = X_block[j_k]
            H_phi = apply_operator_and_compress(H_mpo, ritz_vec, trunc_tol, max_bond)
            lambda_phi = copy.deepcopy(ritz_vec)
            # This creates the -theta * |phi> term
            lambda_phi.A[0] *= -ritz_val 
            # This calculates the residual R = H|phi> - theta|phi>
            residual_vec = add_mps_and_compress(H_phi, lambda_phi, trunc_tol, max_bond)
            R_block.append(residual_vec)
        variances = [vdot(r0, r0) for r0 in R_block]
        variance_history.append(variances)
        print(f"  Current Variances = {variances}")
        # The norm of the residual is sqrt(variance). This is used for convergence.
        residual_norms = [r.orthonormalize('right') for r in R_block]
        print(f"\n  Convergence Check: Residual norms = {np.array(residual_norms)}")
        


        if np.all(np.array(residual_norms) < tol):
            print("\nConvergence criteria met. Stopping.")
            break
        
        if i_restart == num_restarts - 1:
            print("\nMax restarts reached.")
            break

        # --- Phase 3: Augment the Basis (Corrected TRL Step) ---
        print("  Phase 3: Augmenting basis with new Lanczos run from residuals...")
        
        # Start the next basis with the k best Ritz vectors. This is the "Thick" part.
        V_next_basis = copy.deepcopy(X_block)
        
        # Orthonormalize the residual block against the new Ritz vectors and itself
        # to create a set of clean, new directions to start the Lanczos run.
        for r_vec in R_block:
            # Orthogonalize against all vectors currently in the next basis
            for v_prev in V_next_basis:
                overlap = vdot(r_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                r_vec = add_mps_and_compress(r_vec, term, trunc_tol, max_bond)
            
            norm = r_vec.orthonormalize('right')
            if norm > 1e-12:
                # Add the new, clean search direction to the basis
                V_next_basis.append(r_vec)
        
        # If all residuals collapsed, the subspace is invariant, so we stop.
        if len(V_next_basis) == k:
            print("  Residuals collapsed into the existing Ritz space. Stopping.")
            break
        
        # Orthogonalization window size for Lanczos steps
        ortho_window_size = k

        num_generated_so_far = len(V_next_basis) - k
        for j_lan in range(num_generated_so_far, lanczos_steps):
            seed_vec = V_next_basis[-1]
            new_vec = apply_operator_and_compress(H_mpo, seed_vec, trunc_tol, max_bond)
            
            # Define the window of recent vectors to orthogonalize against
            start_index = max(0, len(V_next_basis) - ortho_window_size)
            ortho_basis = V_next_basis[start_index:]
            
            # Orthogonalize against the vectors in the window
            for v_prev in ortho_basis:
                overlap = vdot(new_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                new_vec = add_mps_and_compress(new_vec, term, trunc_tol, max_bond)
                
            norm = new_vec.orthonormalize('right')
            if norm > 1e-12:
                V_next_basis.append(new_vec)
            else:
                print(f"  Lanczos vector {j_lan+1} collapsed. Ending augmentation early.")
                break
        
        V_basis = V_next_basis

    return np.sort(ritz_values[:k]), X_block, np.array(variance_history)

def form_linear_combination(basis, coeffs_vec, max_bond, trunc_tol):
    """(Your helper function remains unchanged)"""
    active_indices = np.where(np.abs(coeffs_vec) > 1e-14)[0]
    if len(active_indices) == 0:
        new_mps = copy.deepcopy(basis[0])
        new_mps.A[0] *= 0
        return new_mps
    start_idx = active_indices[0]
    new_mps = copy.deepcopy(basis[start_idx])
    new_mps.A[0] *= coeffs_vec[start_idx]
    for i in active_indices[1:]:
        term = copy.deepcopy(basis[i])
        term.A[0] *= coeffs_vec[i]
        new_mps = add_mps_and_compress(new_mps, term, trunc_tol, max_bond)
    new_mps.orthonormalize('right')
    return new_mps








In [ ]:
def trl_solver(
    H_mpo,
    k,
    max_bond,
    v0, # Changed from initial_guess_block
    realeig,
    initial_lanczos_steps=20, # New parameter for the initial run
    num_restarts=20,
    lanczos_steps=10,
    tol=1e-12,
    trunc_tol=1e-12
):
    """
    Finds the 'k' lowest eigenpairs using TRL+R, but initializes the
    first basis with a standard Lanczos run.

    Args:
        H_mpo (MPO): The Hamiltonian operator.
        k (int): The number of lowest eigenvalues to find.
        max_bond (int): The maximum bond dimension for the MPS.
        v0 (MPS): A single, normalized MPS to seed the initial Lanczos run.
        realeig (np.ndarray): The true eigenvalues for error tracking.
        initial_lanczos_steps (int): The size of the initial Krylov space.
        num_restarts (int): The number of macro-restart iterations to perform.
        lanczos_steps (int): The number of new vectors to add per restart.
        tol (float): Convergence tolerance for the residual norm.
        trunc_tol (float): Truncation tolerance for MPS operations.

    Returns:
        final_eigs (np.ndarray): The k lowest computed eigenvalues.
        final_vecs (list): A list of k MPS objects for the eigenvectors.
        error_history (np.ndarray): A (num_restarts x k) array of absolute errors.
    """
    error_history = []

    # --- NEW: Phase 0: Initial Lanczos Run ---
    print(f"{'='*20} TRL+R Phase 0: Initial Lanczos Run {'='*20}")
    print(f" Generating an initial basis of size {initial_lanczos_steps}...")
    # Use the Lanczos code to generate the first Krylov basis
    V_basis, _, _, _ = lanczos_lr_py(
        H=H_mpo,
        v0_mps=v0,
        maxiter=initial_lanczos_steps,
        tol=trunc_tol,
        rmax=max_bond
    )
    print(" Initial basis generated.")

    # --- Main Restart Loop ---
    for i_restart in range(num_restarts):
        print(f"\n{'='*20} TRL+R Macro-Iteration {i_restart+1}/{num_restarts} {'='*20}")
        print(f"   Current basis size: {len(V_basis)}")

        # --- Phase 1: Rayleigh-Ritz on the Current Basis ---
        print(f"   Phase 1: Solving Rayleigh-Ritz problem...")
        dim_basis = len(V_basis)
        W = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        P = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        for i in range(dim_basis):
            for j in range(i, dim_basis):
                W[i, j] = vdot(V_basis[i], V_basis[j])
                W[j, i] = np.conj(W[i, j])
                P[i, j] = operator_inner_product(V_basis[i], H_mpo, V_basis[j])
                P[j, i] = np.conj(P[i, j])

        try:
            ritz_values, ritz_coeffs = scipy.linalg.eigh(P, W)
        except scipy.linalg.LinAlgError:
            print("   FATAL: Eigenvalue solve failed on current basis. Stopping.")
            break

        X_block = [form_linear_combination(V_basis, ritz_coeffs[:, j], max_bond, trunc_tol) for j in range(k)]

        # --- Phase 2: Calculate Residuals & Check Convergence ---
        print("   Phase 2: Calculating residuals...")
        R_block = []
        for j_k in range(k):
            ritz_val = ritz_values[j_k]
            ritz_vec = X_block[j_k]
            H_phi = apply_operator_and_compress(H_mpo, ritz_vec, trunc_tol, max_bond)
            lambda_phi = copy.deepcopy(ritz_vec)
            
            # --- FIX #1: Avoid in-place multiplication ---
            lambda_phi.A[0] = lambda_phi.A[0] * -ritz_val
            
            residual_vec = add_mps_and_compress(H_phi, lambda_phi, trunc_tol, max_bond)
            R_block.append(residual_vec)

        residual_norms = [r.orthonormalize('right') for r in R_block]
        print(f"\n   Convergence Check: Residual norms = {np.array(residual_norms)}")

        errors = np.abs(np.sort(ritz_values[:k]) - realeig[:k])
        error_history.append(errors)
        print(f"   Current Errors = {errors}")

        if np.all(np.array(residual_norms) < tol):
            print("\nConvergence criteria met. Stopping.")
            break

        if i_restart == num_restarts - 1:
            print("\nMax restarts reached.")
            break

        # --- Phase 3: Augment the Basis (Corrected TRL Step) ---
        print("   Phase 3: Augmenting basis with new Lanczos run from residuals...")

        V_next_basis = copy.deepcopy(X_block)
        R_ortho_block = []
        for r_vec in R_block:
            for v_prev in V_next_basis:
                overlap = vdot(r_vec, v_prev)
                term = copy.deepcopy(v_prev)
                
                # --- FIX #2: Avoid in-place multiplication ---
                term.A[0] = term.A[0] * -overlap
                
                r_vec = add_mps_and_compress(r_vec, term, trunc_tol, max_bond)

            norm = r_vec.orthonormalize('right')
            if norm > 1e-12:
                V_next_basis.append(r_vec)

        if len(V_next_basis) == k:
            print("   Residuals collapsed into the existing Ritz space. Stopping.")
            break

        ortho_window_size = k
        num_generated_so_far = len(V_next_basis) - k
        for j_lan in range(num_generated_so_far, lanczos_steps):
            seed_vec = V_next_basis[-1]
            new_vec = apply_operator_and_compress(H_mpo, seed_vec, trunc_tol, max_bond)
            
            start_index = max(0, len(V_next_basis) - ortho_window_size)
            ortho_basis = V_next_basis[start_index:]

            for v_prev in ortho_basis:
                overlap = vdot(new_vec, v_prev)
                term = copy.deepcopy(v_prev)
                
                # --- FIX #3: Avoid in-place multiplication ---
                term.A[0] = term.A[0] * -overlap
                
                new_vec = add_mps_and_compress(new_vec, term, trunc_tol, max_bond)

            norm = new_vec.orthonormalize('right')
            if norm > 1e-12:
                V_next_basis.append(new_vec)
            else:
                print(f"   Lanczos vector {j_lan+1} collapsed. Ending augmentation early.")
                break

        V_basis = V_next_basis

    return np.sort(ritz_values[:k]), X_block, np.array(error_history)

# Note: The 'form_linear_combination' function was not provided in the prompt, 
# but the original error traceback pointed to a similar issue inside it. 
# Make sure you apply the same fix there if the error persists. For example:
def form_linear_combination(basis, coeffs_vec, max_bond, trunc_tol):
    # This is a mock-up of the function based on the traceback
    # ... (find active_indices)
    active_indices = [i for i, c in enumerate(coeffs_vec) if abs(c) > 1e-16]
    if not active_indices:
        # Return a zero MPS or handle error
        return basis[0] * 0 
    
    start_idx = active_indices[0]
    new_mps = copy.deepcopy(basis[start_idx])
    
    # The original error likely happened here.
    # We apply the fix preemptively.
    new_mps.A[0] = new_mps.A[0] * coeffs_vec[start_idx]
    
    for i in active_indices[1:]:
        term = copy.deepcopy(basis[i])
        # And also here
        term.A[0] = term.A[0] * coeffs_vec[i]
        new_mps = add_mps_and_compress(new_mps, term, trunc_tol, max_bond)
    
    return new_mps

In [ ]:
from pytenet.global_krylov_method import generate_Hamiltonian_with_occupation_number
e_ground = -84.92262983120877
e_1st_ex = -84.52780562388604
e_2nd_ex = -84.46820215626565
e_3rd_ex = -84.42486181064207
e_4th_ex = -84.42379006099287
realeig = np.array([e_ground, e_1st_ex, e_2nd_ex, e_3rd_ex, e_4th_ex])

H_correct = scipy.io.mmread('data_water/H_water_correct.mtx').tocsr()
H_correct_10e = generate_Hamiltonian_with_occupation_number(H_correct.real, 10)
realeig, real_vec = eigsh(H_correct_10e, k=10, which='SA', return_eigenvectors=True)

opt_err = []
for i in range(10):
    gs_mps = pytenet.MPS.from_vector(d=4, nsites=7, v=real_vec[:,i], tol=1e-12)
    gs_mps.orthonormalize('right')
    gs_mps.compress_direct_svd_left_max_bond(1e-12,24)
    gs_mps.orthonormalize('right')
    opt_err.append(operator_average(gs_mps,mpo_ref) - realeig[i])




In [ ]:
realeig =[ -84.92262983120877, -84.52780562388604, -84.46820215626565, -84.42486181064207, -84.42379006099287]

In [ ]:
opt_err 

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
iterations_run = error_history.shape[0]
for i in range(3):
    ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o')
    
legends =['Ground State', 'First Excited State', 'Second Excited State', 'Third Excited State', 'Fourth Excited State']
ax.set_xlabel("Number of Restart", fontsize=16)
ax.set_ylabel("Absolute Energy Error", fontsize=16)
ax.legend(legends,fontsize=14)
ax.grid(True, which="both", linestyle='--')
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\h2o_trl_new_600.pdf")
plt.show()


In [ ]:
if __name__ == '__main__':
    #mpo_ref =  ptn.heisenberg_xxz_mpo(L = 10, J= -4, D = -4, h= 2)
    #mpo_ref.zero_qnumbers()
    #mat = mpo_ref.as_matrix()
    #realeig, real_vec = np.linalg.eigh(mat)
    #realeig = np.sort(realeig)
    rng_master = np.random.default_rng(42)
    HFS1 = ptn.generate_single_state(7, [3, 3, 3, 3, 1, 2, 0])
    HFS2 = ptn.generate_single_state(7, [3, 3, 2, 3, 1, 2, 1])
    HFS3 = ptn.generate_single_state(7, [3, 3, 3, 3, 3, 0, 0])
    HFS4 = ptn.generate_single_state(7, [3, 0, 0, 3, 3, 3, 3])
    HFS5 = ptn.generate_single_state(7, [3, 3, 3, 0, 0, 3, 3])
  
    h2o_block = [HFS1, HFS2, HFS3,  HFS4, HFS5]


    k = 5
    max_bond = 20
    block = []
    for i in range(k):
        #new_mps = create_random_mps_4([1, 4, 16, 16, 16, 16, 4, 1], 'right', rng=rng_master)
        new_mps = create_random_mps([ 1, 2, 2, 2, 2, 2, 2, 2, 2, 2,2,2 ,1], 'right', rng=rng_master)

        block.append(new_mps)
    realeig = np.sort(realeig)

    # 3. Run the TRL+R Solver
    #final_eigs, final_vecs, error_history = trl_solver(
     #   mpo_ref, k, max_bond, new_mps, realeig, num_restarts=55, lanczos_steps=20)

    final_eigs, final_vecs, error_history = trl_plus_r_solver(
        mpo_ref, k, max_bond, block, realeig, num_restarts=70, lanczos_steps= 20
    )
    
    # 4. Plot Convergence
    print("\nPlotting convergence history...")
    fig, ax = plt.subplots(figsize=(12, 8))
    iterations_run = error_history.shape[0]
    for i in range(3):
        ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o', label=f'State {i}')
    ax.set_xlabel("Number of Restart", fontsize=14)
    ax.set_ylabel("Absolute Energy Error", fontsize=14)
    ax.legend()
    ax.grid(True, which="both", linestyle='--')
    plt.show()
    #plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\h2o_mtrl_HFS.pdf")

In [ ]:
if __name__ == '__main__':
    #mpo_ref =  ptn.heisenberg_xxz_mpo(L = 10, J= -4, D = -4, h= 2)
    #mpo_ref.zero_qnumbers()
    #mat = mpo_ref.as_matrix()
    #realeig, real_vec = np.linalg.eigh(mat)
    #realeig = np.sort(realeig)
    rng_master = np.random.default_rng(42)
    HFS1 = ptn.generate_single_state(7, [3, 3, 3, 3, 1, 2, 0])
    HFS2 = ptn.generate_single_state(7, [3, 3, 2, 3, 1, 2, 1])
    HFS3 = ptn.generate_single_state(7, [3, 3, 3, 3, 3, 0, 0])
    HFS4 = ptn.generate_single_state(7, [3, 0, 0, 3, 3, 3, 3])
    HFS5 = ptn.generate_single_state(7, [3, 3, 3, 0, 0, 3, 3])
  
    h2o_block = [HFS1, HFS2, HFS3,  HFS4, HFS5]


    k = 5
    max_bond = 20
    block = []
    for i in range(k):
        #new_mps = create_random_mps_4([1, 4, 16, 16, 16, 16, 4, 1], 'right', rng=rng_master)
        new_mps = create_random_mps([ 1, 2, 2, 2, 2, 2, 2, 2, 2, 2,2,2 ,1], 'right', rng=rng_master)

        block.append(new_mps)
    realeig = np.sort(realeig)

    # 3. Run the TRL+R Solver
    #final_eigs, final_vecs, error_history = trl_solver(
     #   mpo_ref, k, max_bond, new_mps, realeig, num_restarts=55, lanczos_steps=20)

    final_eigs, final_vecs, error_history = trl_solver(
        mpo_ref, k, max_bond, HFS1, realeig, num_restarts=150, lanczos_steps= 20
    )
    
    # 4. Plot Convergence
    print("\nPlotting convergence history...")
    fig, ax = plt.subplots(figsize=(12, 8))
    iterations_run = error_history.shape[0]
    for i in range(3):
        ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o', label=f'State {i}')
    ax.set_xlabel("Number of Restart", fontsize=14)
    ax.set_ylabel("Absolute Energy Error", fontsize=14)
    ax.legend()
    ax.grid(True, which="both", linestyle='--')
    plt.show()
    #plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\h2o_mtrl_HFS.pdf")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8), dpi=100)
colors = ['b','orange','g','r','purple']
iterations_run = error_history.shape[0]
labels = ['Ground State', 'First Excited State', 'Second Excited State', "Third Excited State", "Fourth Excited State"]
for i in range(5):
    ax.semilogy(range(1, iterations_run * 4 + 1,4), error_history[:, i], marker='o', label= labels[i],color=colors[i])
    ax.axhline(
            y=np.abs(opt_err[i]), 
            color=colors[i], 
            linestyle='--', 
            linewidth=2)
ax.set_xlabel("Number of Krylov Vectors for Each Ritz Vector", fontsize=16)
ax.set_ylabel("Energy Error", fontsize=16)
ax.legend(fontsize=16)
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\h2o_mtrl.pdf", bbox_inches='tight', pad_inches=0.05)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

    # Define a color cycle for the plots
colors = plt.cm.viridis(np.linspace(0, 1, k))


ax.semilogy(
            range(1, iterations_run + 1), 
            error_history[:, 0], 
            marker='o', 
            linestyle='-', 
            color=colors[0],
            markersize=5, 
            label=f'MTRL: Ground State 1')

ax.semilogy(
            range(1, iterations_run + 1), 
            error_history[:, 1], 
            marker='o', 
            linestyle='-', 
            color=colors[1],
            markersize=5, 
            label=f'MTRL: Ground State {2}')

ax.semilogy(
            range(1, iterations_run + 1), 
            error_history[:, 2], 
            marker='o', 
            linestyle='-', 
            color=colors[2],
            markersize=5, 
            label=f'MTRL: First Excited State {1}')

ax.semilogy(
            range(1, iterations_run + 1), 
            error_history[:, 3], 
            marker='o', 
            linestyle='-', 
            color=colors[3],
            markersize=5, 
            label=f'MTRL: First Excited State {2}')

ax.semilogy(
            range(1, iterations_run + 1), 
            error_history[:, 4], 
            marker='o', 
            linestyle='-', 
            color=colors[4],
            markersize=5, 
            label=f'MTRL: Second Excited State')
    

ax.axhline(
            y=np.abs(opt_err[0]), 
            color=colors[0], 
            linestyle='--', 
            linewidth=2,
            label=f'Optimized Error: Ground State 1' )

ax.axhline(
            y=np.abs(opt_err[1]), 
            color=colors[1], 
            linestyle='--', 
            linewidth=2,
            label=f'Optimized Error: Ground State 2' )

ax.axhline(
            y=np.abs(opt_err[2]), 
            color=colors[2], 
            linestyle='--', 
            linewidth=2,
            label=f'Optimized Error: First Excited State 1' )

ax.axhline(
            y=np.abs(opt_err[3]), 
            color=colors[3], 
            linestyle='--', 
            linewidth=2,
            label=f'Optimized Error: First Excited State 2' )

ax.axhline(
            y=np.abs(opt_err[4]), 
            color=colors[4], 
            linestyle='--', 
            linewidth=2,
            label=f'Optimized Error: Second Excited State' )

    # Formatting the plot for a professional look
ax.set_xlabel(" Restart Iteration", fontsize=16, fontweight='bold')
ax.set_ylabel("Energy Error  |E_calc - E_exact|", fontsize=16, fontweight='bold')

    
    # Create a more organized legend
handles, labels = ax.get_legend_handles_labels()
    # Manually create a clean legend
unique_labels = dict(zip(labels, handles))
ax.legend(unique_labels.values(), unique_labels.keys(), fontsize=14)
    
ax.grid(True, which="both", linestyle='--', linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=14)
    
    # Set a reasonable floor for the y-axis to make the plot clean
plt.ylim(bottom=1e-5)
plt.tight_layout()

    # Show the plot
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\trl_convergence_fermi.pdf")
plt.show()
print("Plot generated.")


In [ ]:
if __name__ == '__main__':
    mpo_ref = ptn.bose_hubbard_mpo(d=2,L = 32, U=10, t=1, mu = 1)
    #mpo_ref = ptn.heisenberg_xxz_mpo(L = 36, J= -4, D = -4, h= 2)
    mpo_ref.zero_qnumbers()
    #mat = mpo_ref.as_matrix()
    #realeig, real_vec = np.linalg.eigh(mat)
    #realeig = np.sort(realeig)
    rng_master = np.random.default_rng(42)
    k =6
    max_bond = 64


    ascending = [2**i for i in range(7)]
    descending = ascending[:-1][::-1]
    total_length = 33
    num_middle_elements = total_length - len(ascending) - len(descending)
    middle = [64] * num_middle_elements
    final_list = ascending + middle + descending


    block = []
    for i in range(k):
        #new_mps = create_random_mps_4([1, 4, 16, 20, 20, 16, 4, 1], 'right', rng=rng_master)
        new_mps = create_random_mps(final_list, 'right', rng=rng_master)
        block.append(new_mps)
    realeig = np.sort(realeig)

    # 3. Run the TRL+R Solver
    final_eigs, final_vecs, var_history = trl_plus_r_solver_variance(
        mpo_ref, k, max_bond, block, num_restarts=100, lanczos_steps=30,tol=1e-12, trunc_tol=1e-12
    )
    
    # 4. Plot Convergence
    print("\nPlotting convergence history...")
    fig, ax = plt.subplots(figsize=(12, 8))
    iterations_run = var_history.shape[0]
    for i in range(k):
        ax.semilogy(range(1, iterations_run + 1), var_history[:, i], marker='o', label=f'State {i}')
    ax.set_xlabel("TRL+R Restart Iteration", fontsize=14)
    ax.set_ylabel("Energy Variance", fontsize=14)
    ax.legend()
    ax.grid(True, which="both", linestyle='--')
    plt.show()
    plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\bose_trl_var.pdf")

In [ ]:
  
fig, ax = plt.subplots(figsize=(12, 8))
iterations_run = var_history.shape[0]
for i in range(6):
    ax.semilogy(range(1, iterations_run + 1), var_history[:, i], marker='o')

ax.set_xlabel("Number of Restart", fontsize=16)
ax.set_ylabel("Energy Variance", fontsize=16)
ax.legend(['Ground State', 'First Excited State', 'Second Excited State', 'Third Excited State', 'Fourth Excited State', 'Fifth Excited State'], fontsize=14    )
ax.grid(True, which="both", linestyle='--')
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\bose_long_trl_var.pdf")
plt.show()

In [ ]:
e1 = dmrg_singlesite(H, HFS, 5,30)
e1[-1] - realeig[1]

In [ ]:
print(y_dmrg)
print(y_lan)
print(y_ed)
xx = [6,8,10,12,14,16]

with open('y_dmrg_1st.pkl', 'wb') as f:
    pickle.dump(y_dmrg, f)
with open('y_lan_1st.pkl', 'wb') as f:
    pickle.dump(y_lan, f)
with open('y_ed_1st.pkl', 'wb') as f:
    pickle.dump(y_ed, f)

In [ ]:

plt.figure(figsize=(8, 6),dpi = 100,constrained_layout=True)
plt.plot(xx, y_dmrg, 'o-', label="DMRG")
plt.plot(xx, y_lan, '^-', label="Lanczos")
plt.plot(xx, y_ed, 'v-', label="ED")



plt.xlabel("Max bond dimension of ground state", fontsize=16)
plt.ylabel("Energy error of 1st excited state", fontsize=16)


#plt.ylim(1e-11, 1e1)

ax = plt.gca()
ax.tick_params(axis='both', which='major', labelsize=16)

# Add horizontal line at 1e-7


# Optional: force y-axis ticks at each 10^-x
#yticks = [10**(-i) for i in range(1,12,2)]  # 10^0 to 10^-10
#plt.yticks(yticks)

ax = plt.gca() # Get current axes
ax.tick_params(axis='both', which='major', labelsize=16)
plt.yscale('log')

plt.legend(fontsize=12)
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\compare_lan.pdf",bbox_inches='tight',
        pad_inches=0.05)
plt.show()


In [ ]:
N_Krylov = 30
max_bond_Krylov = 100
e_band = []
#HFS1 = generate_single_state(7, [3, 3, 3, 3, 3, 0, 0])
N_list = [0]
foldername = f"test_gs"
l = 5
trunc_tol = 10e-12
generate_krylov_space(N_Krylov, mpo_ref, HFS, max_bond_Krylov, trunc_tol, foldername)
N_use = 30
# C = coeff_canonical_orthogonalization(N_use, foldername)
# vector_list = generate_re_ortho_space_with_coeff(N_use, C, foldername)
#vector_list = generate_re_ortho_space(N_use, foldername)
H_reduced_non_rotho = generate_reduced_H_non_ortho(N_use, foldername, mpo_ref)
W = get_W(N_use, foldername)
coeff = get_S(W)
coeff = np.array(coeff)
H_reduced = np.einsum('ik, kl, jl -> ij', coeff.conj(), H_reduced_non_rotho, coeff)
e0,initial_block = solve_ritz_vecs(foldername, H_reduced, N_use, coeff, max_bond_Krylov, 7, mpo_ref)
initial_block = initial_block[:l]
e_band.append(np.array(e0))

In [ ]:
num_op = 5
N_Krylov = l * num_op 
num_restart = 20
lock_idx = 0  
H = mpo_ref
err_var_hei = []    
for res in range(num_restart):
    generate_band_krylov(initial_block, N_Krylov, H, max_bond_Krylov, trunc_tol, foldername,lock_idx)
    H_reduced_non_rotho = generate_reduced_H_non_ortho(N_Krylov+l, foldername, H)
    W = get_W(N_Krylov+l, foldername)
    coeff = get_S(W)
    coeff = np.array(coeff)
    H_reduced = np.einsum('ik, kl, jl -> ij', coeff.conj(), H_reduced_non_rotho, coeff)
    e,initial_block = solve_ritz_vecs(foldername, H_reduced, N_Krylov+l, coeff, max_bond_Krylov, 5, H)
    initial_block = initial_block[:l]
    e_band.append(np.array(e))
    err_temp = []
    
    for i in range(l):
        v_test = initial_block[i]
        temp1 = apply_operator(H, v_test)  
        temp2 = add_mps(temp1,v_test,-e_band[-1][i])
        err_temp.append(vdot(temp2,temp2))
    err_var_hei.append(np.array(err_temp))
    print(err_var_hei[-1])

    if err_var_hei[-1][lock_idx] < 1e-8:
        lock_idx += 1
        N_Krylov += (l-lock_idx) * num_op

    print(f"restart {res} completed")

In [ ]:
initial_block0 = copy.deepcopy(initial_block)   
err_var_hei0 = copy.deepcopy(err_var_hei)

In [ ]:

for i in range(l):
    v_test = initial_block0[i]
    temp1 = apply_operator(H, v_test)  
    temp2 = add_mps(temp1,v_test,-e_band[-1][i])
    print(vdot(temp2,temp2))


In [ ]:
xx = np.linspace(30+25,30+25*20, 19)
y0 = [err_var_hei[i][0] for i in range(19)]
y1 = [err_var_hei[i][1]  for i in range(19)]
y2 = [err_var_hei[i][2] for i in range(19)]
y3 = [err_var_hei[i][3]  for i in range(19)]
y4 = [err_var_hei[i][4]  for i in range(19)]



plt.figure(figsize=(8, 6),dpi=100,constrained_layout=True)




plt.plot(xx, y0, 'o-', label="Ground State")
plt.plot(xx, y1, '^-', label="First Excited State")
plt.plot(xx, y2, 'v-', label="Second Excited State")
plt.plot(xx, y3, 'h-', label="Third Excited State")
plt.plot(xx, y4, '*-', label="Fourth Excited State")




plt.xlabel("Number of Krylov vectors", fontsize=16)
plt.ylabel("Energy Variance", fontsize=16)


ax = plt.gca() # Get current axes
ax.tick_params(axis='both', which='major', labelsize=16) # Set size for major ticks on both axes


plt.legend(fontsize=16)  
plt.yscale('log')
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\xxz_30.pdf", bbox_inches='tight',pad_inches=0.05)    
plt.show()

In [ ]:
err_var_hei

In [ ]:
def chebyshev_filter_mps(
    H_mpo,
    mps_in,
    degree,
    filter_a,
    filter_b,
    trunc_tol,
    max_bond
):
    """
    Applies a Chebyshev polynomial filter p(H) to an MPS.

    This function computes y = p(H) * x using a three-term recurrence,
    where the polynomial filter is designed to amplify eigenvalues below 'filter_a'
    and suppress eigenvalues within the interval [filter_a, filter_b].

    Args:
        H_mpo (MPO): The Hamiltonian operator.
        mps_in (MPS): The input MPS vector to be filtered.
        degree (int): The degree of the Chebyshev polynomial.
        filter_a (float): The lower bound of the unwanted spectrum.
        filter_b (float): The upper bound of the unwanted spectrum.
        trunc_tol (float): Truncation tolerance for MPS operations.
        max_bond (int): The maximum bond dimension for the MPS.

    Returns:
        MPS: The filtered MPS object, approximating p(H) * mps_in.
    """
    if degree == 0:
        return mps_in
    if degree == 1:
        return apply_operator_and_compress(H_mpo, mps_in, trunc_tol, max_bond)

    # Define the linear map from [filter_a, filter_b] to [-1, 1]
    c = (filter_b + filter_a) / 2.0
    e = (filter_b - filter_a) / 2.0

    # Three-term recurrence for q_k = p_k(H) * mps_in
    # q_0 = mps_in
    q_prev = copy.deepcopy(mps_in)

    # q_1 = (H*q_0 - c*q_0) / e
    H_q0 = apply_operator_and_compress(H_mpo, q_prev, trunc_tol, max_bond)
    c_q0 = copy.deepcopy(q_prev)
    c_q0.A[0] *= -c
    q_curr = add_mps_and_compress(H_q0, c_q0, trunc_tol, max_bond)
    q_curr.A[0] /= e
    
    # Recurrence for j = 2 to degree
    for _ in range(2, degree + 1):
        # term1 = 2 * (H*q_curr - c*q_curr) / e
        H_q_curr = apply_operator_and_compress(H_mpo, q_curr, trunc_tol, max_bond)
        c_q_curr = copy.deepcopy(q_curr)
        c_q_curr.A[0] *= -c
        term1_num = add_mps_and_compress(H_q_curr, c_q_curr, trunc_tol, max_bond)
        term1_num.A[0] *= (2.0 / e)
        
        # term2 = -q_prev
        term2 = copy.deepcopy(q_prev)
        term2.A[0] *= -1.0
        
        # q_next = term1 + term2
        q_next = add_mps_and_compress(term1_num, term2, trunc_tol, max_bond)

        
        q_prev = q_curr
        q_curr = q_next
        
    return q_curr


def filterlan(
    H_mpo,
    k,
    max_bond,
    initial_guess_block,
    realeig,
    num_restarts=20,
    lanczos_steps=25,
    degree=5,
    a=-1,
    b =0,
    tol=1e-12,
    trunc_tol=1e-12
):
    """
    Finds the 'k' lowest eigenpairs of a Hamiltonian MPO using a
    Thick Restart Lanczos with Residual Enrichment (TRL+R) algorithm.

    This implementation follows the correct TRL procedure where the basis
    is augmented by a new Lanczos run seeded by the residuals.

    Args:
        H_mpo (MPO): The Hamiltonian operator.
        k (int): The number of lowest eigenvalues to find.
        max_bond (int): The maximum bond dimension for the MPS.
        initial_guess_block (list): A list of 'k' MPS objects to start with.
        realeig (np.ndarray): The true eigenvalues for error tracking.
        num_restarts (int): The number of macro-restart iterations to perform.
        lanczos_steps (int): The number of new vectors to generate in each restart cycle.
        tol (float): Convergence tolerance for the residual norm.
        trunc_tol (float): Truncation tolerance for MPS operations.

    Returns:
        final_eigs (np.ndarray): The k lowest computed eigenvalues.
        final_vecs (list): A list of k MPS objects for the eigenvectors.
        error_history (np.ndarray): A (num_restarts x k) array of absolute errors.
    """
    
    # Start with the initial guess as our first basis
    V_basis = copy.deepcopy(initial_guess_block)
    error_history = []

    # --- Main Restart Loop ---
    for i_restart in range(num_restarts):
        print(f"\n{'='*20} TRL+R Macro-Iteration {i_restart+1}/{num_restarts} {'='*20}")
        print(f"  Current basis size: {len(V_basis)}")

        # --- Phase 1: Rayleigh-Ritz on the Current Basis ---
        print(f"  Phase 1: Solving Rayleigh-Ritz problem...")
        dim_basis = len(V_basis)
        W = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        P = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        for i in range(dim_basis):
            for j in range(i, dim_basis):
                W[i, j] = vdot(V_basis[i], V_basis[j])
                W[j, i] = np.conj(W[i, j])
                P[i, j] = operator_inner_product(V_basis[i], H_mpo, V_basis[j])
                P[j, i] = np.conj(P[i, j])
        
        try:
            ritz_values, ritz_coeffs = scipy.linalg.eigh(P, W)
        except scipy.linalg.LinAlgError:
            print("  FATAL: Eigenvalue solve failed on current basis. Stopping.")
            break
        
        X_block = [form_linear_combination(V_basis, ritz_coeffs[:, j], max_bond, trunc_tol) for j in range(k)]

        # --- Phase 2: Calculate Residuals & Check Convergence ---
        print("  Phase 2: Calculating residuals...")
        R_block = []
        for j_k in range(k):
            ritz_val = ritz_values[j_k]
            ritz_vec = X_block[j_k]
            H_phi = chebyshev_filter_mps(H_mpo, ritz_vec,degree,a,b, trunc_tol, max_bond)
            lambda_phi = copy.deepcopy(ritz_vec)
            lambda_phi.A[0] *= -ritz_val
            residual_vec = add_mps_and_compress(H_phi, lambda_phi, trunc_tol, max_bond)
            R_block.append(residual_vec)

        residual_norms = [r.orthonormalize('right') for r in R_block]
        print(f"\n  Convergence Check: Residual norms = {np.array(residual_norms)}")
        
        errors = np.abs(np.sort(ritz_values[:k]) - realeig[:k])
        error_history.append(errors)
        print(f"  Current Errors = {errors}")

        if np.all(np.array(residual_norms) < tol):
            print("\nConvergence criteria met. Stopping.")
            break
        
        if i_restart == num_restarts - 1:
            print("\nMax restarts reached.")
            break

        # --- Phase 3: Augment the Basis (Corrected TRL Step) ---
        print("  Phase 3: Augmenting basis with new Lanczos run from residuals...")
        
        # Start the next basis with the k best Ritz vectors. This is the "Thick" part.
        V_next_basis = copy.deepcopy(X_block)
        
        # Orthonormalize the residual block against the new Ritz vectors and itself
        # to create a set of clean, new directions to start the Lanczos run.
        R_ortho_block = []
        for r_vec in R_block:
            # Orthogonalize against all vectors currently in the next basis
            for v_prev in V_next_basis:
                overlap = vdot(r_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                r_vec = add_mps_and_compress(r_vec, term, trunc_tol, max_bond)
            
            norm = r_vec.orthonormalize('right')
            if norm > 1e-12:
                # Add the new, clean search direction to the basis
                V_next_basis.append(r_vec)
        
        # If all residuals collapsed, the subspace is invariant, so we stop.
        if len(V_next_basis) == k:
            print("  Residuals collapsed into the existing Ritz space. Stopping.")
            break

        # Orthogonalization 
            
        #ortho_window_size = 2 * k
        ortho_window_size = k

        num_generated_so_far = len(V_next_basis) - k
        for j_lan in range(num_generated_so_far, lanczos_steps):
            seed_vec = V_next_basis[-1]
            new_vec = chebyshev_filter_mps(H_mpo, seed_vec,degree,a,b, trunc_tol, max_bond)
            
            # Define the window of recent vectors to orthogonalize against
            start_index = max(0, len(V_next_basis) - ortho_window_size)
            ortho_basis = V_next_basis[start_index:]
            
            # Orthogonalize against the vectors in the window
            for v_prev in ortho_basis:
                overlap = vdot(new_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                new_vec = add_mps_and_compress(new_vec, term, trunc_tol, max_bond)
                
            norm = new_vec.orthonormalize('right')
            if norm > 1e-12:
                V_next_basis.append(new_vec)
            else:
                print(f"  Lanczos vector {j_lan+1} collapsed. Ending augmentation early.")
                break
        
        V_basis = V_next_basis

    return np.sort(ritz_values[:k]), X_block, np.array(error_history)


In [ ]:
mpo_ref =  ptn.heisenberg_xxz_mpo(L = 12, J= 1, D = 1, h= 0)
mpo_ref.zero_qnumbers()
mat = mpo_ref.as_matrix(sparse_format=True)
realeig, real_vec = eigsh(mat, k=10, which='SA')
realeig = np.sort(realeig)
print(realeig)
rng_master = np.random.default_rng(42)

In [ ]:
if __name__ == '__main__':
    mpo_ref =  ptn.heisenberg_xxz_mpo(L = 12, J= 1, D = 1, h= 0)
    mpo_ref.zero_qnumbers()
    mat = mpo_ref.as_matrix()
    realeig, real_vec = eigsh(mat)
    realeig = np.sort(realeig)
    rng_master = np.random.default_rng(42)

  


    k = 5
    max_bond = 16
    block = []
    for i in range(k):
        #new_mps = create_random_mps_4([1, 4, 16, 16, 16, 16, 4, 1], 'right', rng=rng_master)
        new_mps = create_random_mps([ 1, 2, 4, 8, 16,16,16, 16, 16, 8,4,2 ,1], 'right', rng=rng_master)

        block.append(new_mps)
    realeig = np.sort(realeig)

    # 3. Run the TRL+R Solver
    #final_eigs, final_vecs, error_history = trl_solver(
     #   mpo_ref, k, max_bond, new_mps, realeig, num_restarts=55, lanczos_steps=20)

    final_eigs, final_vecs, error_history = filterlan(
    mpo_ref,
    k,
    max_bond,
    block,
    realeig,
    num_restarts=20,
    lanczos_steps=25,
    tol=1e-12,
    trunc_tol=1e-12,
    degree=8,
    a=-4, # This is 'a': the upper bound of the desired spectrum
    b= 2.75     # This is 'b': the top of the full spectrum
)
    
    # 4. Plot Convergence
    print("\nPlotting convergence history...")
    fig, ax = plt.subplots(figsize=(12, 8))
    iterations_run = error_history.shape[0]
    for i in range(k):
        ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o', label=f'State {i}')
    ax.set_xlabel("Number of Restart", fontsize=14)
    ax.set_ylabel("Absolute Energy Error", fontsize=14)
    ax.legend()
    ax.grid(True, which="both", linestyle='--')
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
iterations_run = error_history.shape[0]
for i in range(5):
    ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o')

xticks = [i for i in range(0,22,2)]  # 10^0 to 10^-10
plt.xticks(xticks)


ax.set_xlabel("Number of Restart", fontsize=16)
ax.set_ylabel("Energy Error", fontsize=16)
ax.legend(['Ground State', 'First Excited State', 'Second Excited State', 'Third Excited State', 'Fourth Excited State'], fontsize=14    )
ax.grid(True, which="both", linestyle='--')
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\filterlan_compare_3.pdf")
plt.show()

In [ ]:
import numpy as np
from scipy.sparse.linalg import gmres, LinearOperator

# Assuming MPO, MPS, and apply_operator are defined elsewhere as provided.

def apply_inverse_shifted_mpo(
    H: MPO,
    psi: MPS,
    shift: float,
    tol: float = 1e-8,
    max_bond = 20
) -> MPS:

    L = psi.nsites
    d = len(psi.qd)

    # 1. Construct the shifted operator MPO: (H - shift * I)
    ident_mpo = MPO.identity(qd=H.qd, L=L, dtype=np.complex128)
    ident_mpo.A[0] *= -shift  # Scale the identity MPO by the shift value
    H_shifted = add_mpo(H, ident_mpo) 

    psi_vector = psi.as_vector()
    inv = np.linalg.inv(H_shifted.as_matrix())
    result_vector = inv @ psi_vector
    result_mps = MPS.from_vector(2,H.nsites,result_vector, tol)
    result_mps.compress_no_normalization_max_bond(tol,max_bond)
    return result_mps

In [ ]:
mpo_ref =  ptn.heisenberg_xxz_mpo(L = 10, J= 1, D = 1, h= 0)
mpo_ref.zero_qnumbers()
mat = mpo_ref.as_matrix()
sigma = 0.2
psi = new_mps = create_random_mps([ 1, 2, 4, 8, 16, 16, 16, 8,4,2 ,1], 'right', rng=rng_master)

In [ ]:
def shift_invert_lan(
    H_mpo,
    k,
    max_bond,
    initial_guess_block,
    realeig,
    sigma,
    num_restarts=20,
    lanczos_steps=25,
    tol=1e-12,
    trunc_tol=1e-12
):
    """
    Finds the 'k' lowest eigenpairs of a Hamiltonian MPO using a
    Thick Restart Lanczos with Residual Enrichment (TRL+R) algorithm.

    This implementation follows the correct TRL procedure where the basis
    is augmented by a new Lanczos run seeded by the residuals.

    Args:
        H_mpo (MPO): The Hamiltonian operator.
        k (int): The number of lowest eigenvalues to find.
        max_bond (int): The maximum bond dimension for the MPS.
        initial_guess_block (list): A list of 'k' MPS objects to start with.
        realeig (np.ndarray): The true eigenvalues for error tracking.
        num_restarts (int): The number of macro-restart iterations to perform.
        lanczos_steps (int): The number of new vectors to generate in each restart cycle.
        tol (float): Convergence tolerance for the residual norm.
        trunc_tol (float): Truncation tolerance for MPS operations.

    Returns:
        final_eigs (np.ndarray): The k lowest computed eigenvalues.
        final_vecs (list): A list of k MPS objects for the eigenvectors.
        error_history (np.ndarray): A (num_restarts x k) array of absolute errors.
    """
    
    # Start with the initial guess as our first basis
    V_basis = copy.deepcopy(initial_guess_block)
    error_history = []

    # --- Main Restart Loop ---
    for i_restart in range(num_restarts):
        print(f"\n{'='*20} TRL+R Macro-Iteration {i_restart+1}/{num_restarts} {'='*20}")
        print(f"  Current basis size: {len(V_basis)}")

        # --- Phase 1: Rayleigh-Ritz on the Current Basis ---
        print(f"  Phase 1: Solving Rayleigh-Ritz problem...")
        dim_basis = len(V_basis)
        W = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        P = np.zeros((dim_basis, dim_basis), dtype=np.complex128)
        for i in range(dim_basis):
            for j in range(i, dim_basis):
                W[i, j] = vdot(V_basis[i], V_basis[j])
                W[j, i] = np.conj(W[i, j])
                P[i, j] = operator_inner_product(V_basis[i], H_mpo, V_basis[j])
                P[j, i] = np.conj(P[i, j])
        
        try:
            ritz_values, ritz_coeffs = scipy.linalg.eigh(P, W)
        except scipy.linalg.LinAlgError:
            print("  FATAL: Eigenvalue solve failed on current basis. Stopping.")
            break
        
        X_block = [form_linear_combination(V_basis, ritz_coeffs[:, j], max_bond, trunc_tol) for j in range(k)]

        # --- Phase 2: Calculate Residuals & Check Convergence ---
        print("  Phase 2: Calculating residuals...")
        R_block = []
        for j_k in range(k):
            ritz_val = ritz_values[j_k]
            ritz_vec = X_block[j_k]
            H_phi = apply_inverse_shifted_mpo(H_mpo, ritz_vec,sigma, trunc_tol, max_bond)
            lambda_phi = copy.deepcopy(ritz_vec)
            lambda_phi.A[0] *= -ritz_val
            residual_vec = add_mps_and_compress(H_phi, lambda_phi, trunc_tol, max_bond)
            R_block.append(residual_vec)

        residual_norms = [r.orthonormalize('right') for r in R_block]
        print(f"\n  Convergence Check: Residual norms = {np.array(residual_norms)}")
        
        errors = np.abs(np.sort(ritz_values[:k]) - realeig[:k])
        error_history.append(errors)
        print(f"  Current Errors = {errors}")

        if np.all(np.array(residual_norms) < tol):
            print("\nConvergence criteria met. Stopping.")
            break
        
        if i_restart == num_restarts - 1:
            print("\nMax restarts reached.")
            break

        # --- Phase 3: Augment the Basis (Corrected TRL Step) ---
        print("  Phase 3: Augmenting basis with new Lanczos run from residuals...")
        
        # Start the next basis with the k best Ritz vectors. This is the "Thick" part.
        V_next_basis = copy.deepcopy(X_block)
        
        # Orthonormalize the residual block against the new Ritz vectors and itself
        # to create a set of clean, new directions to start the Lanczos run.
        R_ortho_block = []
        for r_vec in R_block:
            # Orthogonalize against all vectors currently in the next basis
            for v_prev in V_next_basis:
                overlap = vdot(r_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                r_vec = add_mps_and_compress(r_vec, term, trunc_tol, max_bond)
            
            norm = r_vec.orthonormalize('right')
            if norm > 1e-12:
                # Add the new, clean search direction to the basis
                V_next_basis.append(r_vec)
        
        # If all residuals collapsed, the subspace is invariant, so we stop.
        if len(V_next_basis) == k:
            print("  Residuals collapsed into the existing Ritz space. Stopping.")
            break

        # Orthogonalization 
            
        #ortho_window_size = 2 * k
        ortho_window_size = k

        num_generated_so_far = len(V_next_basis) - k
        for j_lan in range(num_generated_so_far, lanczos_steps):
            seed_vec = V_next_basis[-1]
            new_vec = apply_inverse_shifted_mpo(H_mpo, seed_vec,sigma, trunc_tol, max_bond)
            
            # Define the window of recent vectors to orthogonalize against
            start_index = max(0, len(V_next_basis) - ortho_window_size)
            ortho_basis = V_next_basis[start_index:]
            
            # Orthogonalize against the vectors in the window
            for v_prev in ortho_basis:
                overlap = vdot(new_vec, v_prev)
                term = copy.deepcopy(v_prev)
                term.A[0] *= -overlap
                new_vec = add_mps_and_compress(new_vec, term, trunc_tol, max_bond)
                
            norm = new_vec.orthonormalize('right')
            if norm > 1e-12:
                V_next_basis.append(new_vec)
            else:
                print(f"  Lanczos vector {j_lan+1} collapsed. Ending augmentation early.")
                break
        
        V_basis = V_next_basis

    return np.sort(ritz_values[:k]), X_block, np.array(error_history)


In [ ]:
if __name__ == '__main__':
    mpo_ref =  ptn.bose_hubbard_mpo(d=2,L = 10, t= 1, U = 1, mu= 0)
    mpo_ref.zero_qnumbers()
    mat = mpo_ref.as_matrix()
    realeig, real_vec = eigsh(mat, k=10, which='SA')
    realeig = np.sort(realeig)
    rng_master = np.random.default_rng(42)

  


    k = 5
    max_bond = 16
    block = []
    for i in range(k):
        #new_mps = create_random_mps_4([1, 4, 16, 16, 16, 16, 4, 1], 'right', rng=rng_master)
        new_mps = create_random_mps([ 1, 2, 4, 8, 16, 16, 16, 8,4,2 ,1], 'right', rng=rng_master)

        block.append(new_mps)
    realeig = np.sort(realeig)

    # 3. Run the TRL+R Solver
    #final_eigs, final_vecs, error_history = trl_solver(
     #   mpo_ref, k, max_bond, new_mps, realeig, num_restarts=55, lanczos_steps=20)

    final_eigs, final_vecs, error_history = trl_plus_r_solver(
    mpo_ref,
    k,
    max_bond,
    block,
    realeig,
    num_restarts=25,
    lanczos_steps=25,
    tol=1e-12,
    trunc_tol=1e-12
)
    
    # 4. Plot Convergence
    print("\nPlotting convergence history...")
    fig, ax = plt.subplots(figsize=(12, 8))
    iterations_run = error_history.shape[0]
    for i in range(k):
        ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o', label=f'State {i}')
    ax.set_xlabel("Number of Restart", fontsize=14)
    ax.set_ylabel("Absolute Energy Error", fontsize=14)
    ax.legend()
    ax.grid(True, which="both", linestyle='--')
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
iterations_run = error_history.shape[0]
for i in range(5):
    ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o')

xticks = [i for i in range(0,22,2)]  # 10^0 to 10^-10
plt.xticks(xticks)


ax.set_xlabel("Number of Restart", fontsize=16)
ax.set_ylabel("Energy Error", fontsize=16)
ax.legend(['Ground State', 'First Excited State', 'Second Excited State', 'Third Excited State', 'Fourth Excited State'], fontsize=14    )
ax.grid(True, which="both", linestyle='--')
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\filterlan_compare_3.pdf")
plt.show()

In [ ]:
if __name__ == '__main__':
    mpo_ref =  ptn.bose_hubbard_mpo(d=2,L = 10, t= 1, U = 1, mu= 0)
    mpo_ref.zero_qnumbers()
    mat = mpo_ref.as_matrix()
    realeig, real_vec = eigsh(mat, k=10, which='SA')
    realeig = np.sort(realeig)
    rng_master = np.random.default_rng(42)

  


    k = 5
    max_bond = 16
    block = []
    for i in range(k):
        #new_mps = create_random_mps_4([1, 4, 16, 16, 16, 16, 4, 1], 'right', rng=rng_master)
        new_mps = create_random_mps([ 1, 2, 4, 8, 16, 16, 16, 8,4,2 ,1], 'right', rng=rng_master)

        block.append(new_mps)
    realeig = np.sort(realeig)

    # 3. Run the TRL+R Solver
    #final_eigs, final_vecs, error_history = trl_solver(
     #   mpo_ref, k, max_bond, new_mps, realeig, num_restarts=55, lanczos_steps=20)

    final_eigs, final_vecs, error_history = shift_invert_lan(
    mpo_ref,
    k,
    max_bond,
    block,
    realeig,
    sigma=-5.7,
    num_restarts=20,
    lanczos_steps=25,
    tol=1e-12,
    trunc_tol=1e-12
)
    
    # 4. Plot Convergence
    print("\nPlotting convergence history...")
    fig, ax = plt.subplots(figsize=(12, 8))
    iterations_run = error_history.shape[0]
    for i in range(k):
        ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o', label=f'State {i}')
    ax.set_xlabel("Number of Restart", fontsize=14)
    ax.set_ylabel("Absolute Energy Error", fontsize=14)
    ax.legend()
    ax.grid(True, which="both", linestyle='--')
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
iterations_run = error_history.shape[0]
for i in range(5):
    ax.semilogy(range(1, iterations_run + 1), error_history[:, i], marker='o')

xticks = [i for i in range(1,26,2)]  # 10^0 to 10^-10
plt.xticks(xticks)


ax.set_xlabel("Number of Restart", fontsize=16)
ax.set_ylabel("Energy Error", fontsize=16)
ax.legend(['Ground State', 'First Excited State', 'Second Excited State', 'Third Excited State', 'Fourth Excited State'], fontsize=14    )
ax.grid(True, which="both", linestyle='--')
plt.savefig(r"C:\Users\Zhangyu\Downloads\plot lanczos\shift-and-invert-lan_compare.pdf")
plt.show()

In [ ]:
def check_canonical_mps(mps, mode='right', atol=1e-10, ignore_boundary=True):
    """
    mode='right' : checks (Q†Q≈I) for sites 0..L-2 on matrices reshaped as (d*Dl, Dr).
    mode='left'  : checks (Q Q†≈I) for sites 1..L-1 on matrices reshaped as (d*Dr, Dl).
    Returns (ok, max_err).
    """
    L = mps.nsites
    errs = []

    if mode == 'right':
        idxs = range(L-1) if ignore_boundary else range(L)
        for i in idxs:
            d, Dl, Dr = mps.A[i].shape
            M = mps.A[i].reshape(d*Dl, Dr)
            G = M.conj().T @ M
            I = np.eye(Dr, dtype=G.dtype)
            errs.append(np.linalg.norm(G - I))
    elif mode == 'left':
        idxs = range(1, L) if ignore_boundary else range(L)
        for i in idxs:
            d, Dl, Dr = mps.A[i].shape
            M = mps.A[i].transpose(0,2,1).reshape(d*Dr, Dl)
            G = M @ M.conj().T
            I = np.eye(M.shape[0], dtype=G.dtype)
            errs.append(np.linalg.norm(G - I))
    else:
        raise ValueError("mode must be 'right' or 'left'")

    max_err = max(errs) if errs else 0.0
    return (max_err <= atol, max_err)

import numpy as np

def _is_complex_like(arrs):
    return any(np.iscomplexobj(x) for x in arrs)

def _ensure_rng(rng):
    return rng if rng is not None else np.random.default_rng()

def apply_operator_src(
    H_mpo: "MPO",
    psi_mps: "MPS",
    *,
    max_bond: int,
    oversample: int = 8,
    rng=None,
    power_iters: int = 1,          # 0/1/2 is usually enough
    energy_tol: float = 1e-12,     # sketch energy retained (Σσ^2)
    rel_cut: float = 1e-14,        # relative σ cutoff
    normalize_last: bool = False   # keep amplitude unless you explicitly want it
) -> "MPS":
    """
    Single-pass SRC (left→right), QR variant.
    - Wi: (o, i, L, R)  MPO tensors
    - Ai: (i, l, r)     MPS tensors

    Output is left-canonical; no SVD-based polishing is used.
    """
    rng = _ensure_rng(rng)
    Lsites = H_mpo.nsites
    assert Lsites == psi_mps.nsites, "MPO/MPS length mismatch"

    W = H_mpo.A
    A = psi_mps.A

    # Basic checks (OBC)
    for i in range(Lsites):
        do, di, DLH, DRH = W[i].shape
        di2, DLP, DRP    = A[i].shape
        if di != di2:
            raise ValueError(f"Physical mismatch at site {i}: MPO d_in={di}, MPS d={di2}")
        if i == 0 and not (DLH == 1 and DLP == 1):
            raise ValueError(f"Site 0 must have DLH=1 and DLP=1, got DLH={DLH}, DLP={DLP}")
        if i == Lsites - 1 and not (DRH == 1 and DRP == 1):
            raise ValueError(f"Last site must have DRH=1 and DRP=1, got DRH={DRH}, DRP={DRP}")

    complex_like = _is_complex_like([*W, *A])
    dtype = np.complex128 if complex_like else np.float64

    # Left boundary "cap": CapL[L, l, x], start with δ_{L=0,l=0,x=0}
    CapL = np.zeros((W[0].shape[2], A[0].shape[1], 1), dtype=dtype)  # (L, l, x)
    CapL[0, 0, 0] = 1.0
    chi_left = 1

    eta = [None] * Lsites

    # Adjoint matvec for local power-iterations:
    #  given Y[o,x,p], return Z[R,r,p] for current site i
    def _adjoint_mv_L2R(Wi, Ai, CapL, Y):
        # T2b[o,L,l,p] = sum_x conj(CapL[L,l,x]) * Y[o,x,p]
        T2b = np.einsum('Llx,oxp->oLlp', np.conj(CapL), Y, optimize=True)      # (o,L,l,p)
        # T1b[i,l,R,p] = sum_{o,L} conj(Wi[o,i,L,R]) * T2b[o,L,l,p]
        T1b = np.einsum('oiLR,oLlp->ilRp', np.conj(Wi), T2b, optimize=True)    # (i,l,R,p)
        # Z[R,r,p]     = sum_{i,l} conj(Ai[i,l,r]) * T1b[i,l,R,p]
        Z   = np.einsum('ilr,ilRp->Rrp', np.conj(Ai), T1b, optimize=True)      # (R,r,p)
        return Z

    # -------- Left-to-right sweep for all but the final site --------
    for i in range(Lsites - 1):
        Wi = W[i]      # (o, i, L, R)
        Ai = A[i]      # (i, l, r)
        do, di, DLH, DRH = Wi.shape
        _, DLP, DRP      = Ai.shape

        # number of probing columns
        p_max = DRH * DRP
        p = min(max_bond + oversample, p_max)

        # random Ω[R,r,p] and orthonormalise its columns for stability
        if complex_like:
            Om = (rng.standard_normal((DRH, DRP, p)) +
                  1j * rng.standard_normal((DRH, DRP, p))).astype(dtype)
        else:
            Om = rng.standard_normal((DRH, DRP, p)).astype(dtype)
        Qom, _ = np.linalg.qr(Om.reshape(DRH * DRP, p), mode='reduced')
        Om = Qom.reshape(DRH, DRP, Qom.shape[1])
        p = Om.shape[2]

        # Forward matvec: Y[o,x,p] = (CapL ∘ Wi ∘ Ai) Om
        T1 = np.einsum('ilr,Rrp->ilRp', Ai, Om,        optimize=True)  # (i,l,R,p)
        T2 = np.einsum('oiLR,ilRp->oLlp', Wi, T1,      optimize=True)  # (o,L,l,p)
        Y  = np.einsum('Llx,oLlp->oxp',   CapL, T2,    optimize=True)  # (o,x,p)

        # Optional local power iterations on the *small* sketch
        for _ in range(power_iters):
            Z  = _adjoint_mv_L2R(Wi, Ai, CapL, Y)                          # (R,r,p)
            Qz, _ = np.linalg.qr(Z.reshape(DRH * DRP, p), mode='reduced')
            Z  = Qz.reshape(DRH, DRP, Qz.shape[1]); p = Z.shape[2]         # re-orthonormalise probes
            T1 = np.einsum('ilr,Rrp->ilRp', Ai, Z,       optimize=True)
            T2 = np.einsum('oiLR,ilRp->oLlp', Wi, T1,    optimize=True)
            Y  = np.einsum('Llx,oLlp->oxp',   CapL, T2,  optimize=True)

        # QR on the sketch rows, then tiny SVD on R to pick adaptive rank
        Y_mat = Y.reshape(do * chi_left, p)                                 # rows=(o*χL)
        Qy, Ry = np.linalg.qr(Y_mat, mode='reduced')                        # (rows×p), (p×p)
        Uy, Sy, _ = np.linalg.svd(Ry, full_matrices=False)                  # tiny SVD

        if Sy.size == 0:
            k = 1
            Q_use = Qy[:, :k]
        else:
            # relative & energy cut
            keep_rel = np.where(Sy >= Sy[0] * rel_cut)[0]
            k_rel = (int(keep_rel[-1]) + 1) if keep_rel.size else 1
            e = Sy ** 2
            ce = np.cumsum(e) / np.sum(e)
            k_eng = int(np.searchsorted(ce, 1.0 - energy_tol)) + 1
            k = min(max_bond, max(1, min(k_rel, k_eng, Sy.size)))
            Q_use = Qy @ Uy[:, :k]

        eta[i] = Q_use.reshape(do, chi_left, k)                             # left-canonical core

        # Update left environment: CapL' = projection of rows onto span(eta[i])
        U5    = np.einsum('oiLR,ilr->oLRlr', Wi, Ai, optimize=True)        # (o,L,R,l,r)
        Mrow  = np.einsum('Llx,oLRlr->oxRr', CapL, U5, optimize=True)      # (o,x,R,r)
        S     = np.einsum('oxa,oxRr->aRr', np.conj(eta[i]), Mrow, optimize=True)  # (a,R,r)
        CapL  = np.transpose(S, (1, 2, 0)).copy()                           # (R,r,a)
        chi_left = k

    # -------- Last site (exact contraction, no sketch) --------
    Wlast = W[Lsites - 1]    # (o, i, L, 1)
    Alast = A[Lsites - 1]    # (i, l, 1)
    do, di, DLH, DRH = Wlast.shape
    _, DLP, DRP      = Alast.shape
    assert DRH == 1 and DRP == 1
    if not (DLH == CapL.shape[0] and DLP == CapL.shape[1]):
        raise ValueError(
            "Last CapL dims do not match last MPO/MPS left bonds: "
            f"CapL={(CapL.shape[0], CapL.shape[1])}, "
            f"MPO_L={DLH}, MPS_L={DLP}"
        )

    Ulast = np.einsum('oiLr,ilr->oLl', Wlast, Alast, optimize=True)        # (o,L,l)
    Ylast = np.einsum('Llx,oLl->ox',   CapL,  Ulast, optimize=True)        # (o,χL)

    if normalize_last:
        nrm = np.linalg.norm(Ylast)
        if nrm > 0:
            Ylast = Ylast / nrm

    eta[Lsites - 1] = Ylast.reshape(do, chi_left, 1)

    # -------- Build MPS with your constructor (qnumbers disabled) --------
    d_out = do
    bond_dims = [eta[0].shape[1]] + [eta[i].shape[2] for i in range(Lsites)]
    qd = np.zeros(d_out, dtype=int)
    qD = [np.zeros(Di, dtype=int) for Di in bond_dims]
    out = psi_mps.__class__(qd, qD, fill='postpone')
    out.A = eta
    return out


In [ ]:
#A = ptn.ising_mpo(L=12, J=-4, h=2,g = 1)
#psi = create_random_mps([1, 2, 4, 8, 16, 32,64,64, 32, 16, 4, 2, 1], 'right', rng=rng_master)
#A.zero_qnumbers(); psi.zero_qnumbers()


A = ptn.fermi_hubbard_mpo(L=8, t=1, U=8, mu=0)
A.zero_qnumbers()
psi = create_random_mps_4([1, 4, 16, 64,256, 64, 16, 4, 1], 'right', rng=rng_master)
#rng = np.random.default_rng(42)







In [ ]:
phi = apply_operator_src(
    H_mpo=A,
    psi_mps=psi,
    max_bond=50,
    oversample=12,
    rng=rng_master,
    power_iters=2,
    normalize_last=True,
)





In [ ]:
phi_exact = apply_operator(A, psi)

In [ ]:
print(phi.bond_dims)
phi_exact.orthonormalize('right')
print(phi_exact.bond_dims)
vdot(phi, phi_exact)

In [ ]:
phi = apply_operator_and_compress(
    A, psi,1e-12,50)


In [ ]:
print(phi.bond_dims)
phi_exact = apply_operator(A, psi)
phi_exact.orthonormalize('right')
phi.orthonormalize('right')
vdot(phi, phi_exact)